In [1]:
# Python runtime preflight. This notebook was executed with Python 3.9.6.
import shutil
import subprocess
import sys

TARGET_PYTHON_VERSION = "3.9.6"
TARGET_PYTHON_INFO = tuple(int(part) for part in TARGET_PYTHON_VERSION.split("."))
TARGET_KERNEL_NAME = "assignment_2-rag-py396"
TARGET_KERNEL_DISPLAY_NAME = "Python 3.9.6 (assignment_2-rag)"

def _python_version(executable: str):
    result = subprocess.run(
        [executable, "-c", "import sys; print('.'.join(map(str, sys.version_info[:3])))"],
        text=True,
        capture_output=True,
    )
    if result.returncode != 0:
        return None
    return result.stdout.strip()

def _find_python_396():
    candidates = []
    for name in ("python3.9", "python3"):
        executable = shutil.which(name)
        if executable and executable not in candidates:
            candidates.append(executable)

    uv = shutil.which("uv")
    if uv:
        result = subprocess.run([uv, "python", "find", TARGET_PYTHON_VERSION], text=True, capture_output=True)
        if result.returncode == 0:
            executable = result.stdout.strip()
            if executable and executable not in candidates:
                candidates.append(executable)

    for executable in candidates:
        if _python_version(executable) == TARGET_PYTHON_VERSION:
            return executable
    return None

current_python_version = ".".join(map(str, sys.version_info[:3]))
if sys.version_info[:3] == TARGET_PYTHON_INFO:
    print(f"Using required Python {TARGET_PYTHON_VERSION}: {sys.executable}")
else:
    python_396 = _find_python_396()
    uv = shutil.which("uv")
    if python_396 is None and uv:
        print(f"Python {TARGET_PYTHON_VERSION} was not found; installing it with uv.")
        subprocess.run([uv, "python", "install", TARGET_PYTHON_VERSION], check=True)
        python_396 = _find_python_396()

    if python_396 is None:
        raise RuntimeError(
            f"This notebook requires Python {TARGET_PYTHON_VERSION}. Current kernel is {current_python_version}. "
            "Install uv and rerun this cell, or install Python 3.9.6 manually and select that kernel."
        )

    subprocess.run([python_396, "-m", "pip", "install", "-q", "ipykernel"], check=True)
    subprocess.run(
        [
            python_396,
            "-m",
            "ipykernel",
            "install",
            "--user",
            "--name",
            TARGET_KERNEL_NAME,
            "--display-name",
            TARGET_KERNEL_DISPLAY_NAME,
        ],
        check=True,
    )
    raise RuntimeError(
        f"Python {TARGET_PYTHON_VERSION} is available at {python_396} and was registered as "
        f"'{TARGET_KERNEL_DISPLAY_NAME}'. Restart this notebook with that kernel and run from the top."
    )


Using required Python 3.9.6: .venv/bin/python


In [2]:
import re
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

REQUIRED_PACKAGES = [
    "urllib3<2",
    "jupyterlab",
    "ipykernel",
    "ipywidgets",
    "pandas",
    "openpyxl",
    "python-dotenv",
    "tqdm",
    "openai",
    "fsspec",
    "huggingface-hub",
    "langchain-community",
    "langchain-huggingface",
    "langchain-text-splitters",
    "faiss-cpu",
    "sentence-transformers",
    "pypdf",
    "cryptography",
    "ragas",
    "eval_type_backport",
]


def requirement_is_satisfied(requirement: str) -> bool:
    match = re.match(r"^([A-Za-z0-9_.-]+)(.*)$", requirement)
    if not match:
        return False
    dist_name, spec = match.groups()
    try:
        installed_version = version(dist_name)
    except PackageNotFoundError:
        return False
    if spec == "<2":
        return installed_version.split(".", 1)[0].isdigit() and int(installed_version.split(".", 1)[0]) < 2
    return True


missing_requirements = [requirement for requirement in REQUIRED_PACKAGES if not requirement_is_satisfied(requirement)]
if missing_requirements:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_requirements], check=True)
    print(f"Installed missing notebook packages: {missing_requirements}")
else:
    print("All required notebook packages are already installed in the active kernel.")


All required notebook packages are already installed in the active kernel.


In [3]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from getpass import getpass
from pathlib import Path

import ast
import json
import math
import os
import platform
import queue
import re
import shutil
import time
import urllib.parse
import urllib.request
from threading import Lock, Thread

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import certifi
import pandas as pd
import torch
from dotenv import load_dotenv
from IPython.display import display
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import AsyncOpenAI, OpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import Faithfulness
from sentence_transformers import CrossEncoder
from tqdm import tqdm


In [4]:
CURRENT_DIR = Path.cwd()
if CURRENT_DIR.name == "rag_faiss":
    REPO_ROOT = CURRENT_DIR.parent
    FAISS_ROOT = CURRENT_DIR
else:
    REPO_ROOT = CURRENT_DIR
    FAISS_ROOT = CURRENT_DIR / "rag_faiss"

DATA_DIR = REPO_ROOT / "data"
ARTIFACT_DIR = FAISS_ROOT / "artifacts"
OUTPUT_DIR = FAISS_ROOT / "outputs"

try:
    load_dotenv(dotenv_path=REPO_ROOT / ".env", override=False)
except Exception:
    pass

# Environment and artifact paths
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

# Dataset preparation
DATASET_URL = "hf://datasets/PatronusAI/financebench/financebench_merged.jsonl"
DATASET_FALLBACK_URL = "https://huggingface.co/datasets/PatronusAI/financebench/resolve/main/financebench_merged.jsonl"
PDF_REPO_BASE = "https://github.com/patronus-ai/financebench/blob/main/pdfs"
PDF_RAW_BASE = "https://raw.githubusercontent.com/patronus-ai/financebench/main/pdfs"
KEEP_QUESTION_TYPES = {"domain-relevant", "novel-generated"}
FILTERED_DATASET_PATH = DATA_DIR / "financebench_filtered.jsonl"
PDF_DIR = DATA_DIR / "financebench_pdfs"

# Task 1
NEBIUS_BASE_URL = "https://api.tokenfactory.nebius.com/v1/"
API_MAX_RETRIES = 6
API_RETRY_BASE_SECONDS = 2
API_RETRY_MAX_SECONDS = 60
NAIVE_GENERATION_MODEL = "meta-llama/Llama-3.3-70B-Instruct"
NAIVE_SYSTEM_PROMPT = """You are a financial question-answering assistant.
Answer the user's question directly from the information in the question and your general knowledge.
If the question requires company filing data that is not provided, say that the answer cannot be determined from the question alone.
Keep the answer concise and do not invent financial statement values."""
TASK1_RAW_PATH = ARTIFACT_DIR / "task1_naive_generation_raw.json"
TASK1_XLSX_PATH = OUTPUT_DIR / "assignment_2_naive_generation.xlsx"
FORCE_TASK1_REGEN = False

def call_with_retries(operation, description: str, max_retries: int = API_MAX_RETRIES):
    last_error = None
    for attempt in range(max_retries):
        try:
            return operation()
        except Exception as exc:
            last_error = exc
            if attempt == max_retries - 1:
                break
            wait_seconds = min(API_RETRY_MAX_SECONDS, API_RETRY_BASE_SECONDS * (2 ** attempt))
            time.sleep(wait_seconds)
    raise RuntimeError(f"{description} failed after {max_retries} attempts") from last_error

def ensure_api_ca_bundle() -> str:
    ca_bundle_path = DATA_DIR / "cacert.pem"
    ca_bundle_path.parent.mkdir(parents=True, exist_ok=True)
    if not ca_bundle_path.exists() or ca_bundle_path.stat().st_size == 0:
        shutil.copyfile(certifi.where(), ca_bundle_path)
    ca_bundle = str(ca_bundle_path.resolve())
    os.environ["SSL_CERT_FILE"] = ca_bundle
    os.environ["REQUESTS_CA_BUNDLE"] = ca_bundle
    os.environ["CURL_CA_BUNDLE"] = ca_bundle
    return ca_bundle

ensure_api_ca_bundle()

def total_system_ram_gb() -> float:
    try:
        return os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE") / (1024 ** 3)
    except (AttributeError, OSError, ValueError):
        return 0.0

def mps_is_available() -> bool:
    return hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

def choose_local_torch_device(min_mps_ram_gb: int = 64) -> str:
    requested_device = os.getenv("ASSIGNMENT_2_TORCH_DEVICE", "").strip().lower()
    if requested_device in {"cpu", "cuda", "mps"}:
        if requested_device == "cuda" and not torch.cuda.is_available():
            print("ASSIGNMENT_2_TORCH_DEVICE=cuda requested but CUDA is unavailable; falling back to auto selection.")
        elif requested_device == "mps" and not mps_is_available():
            print("ASSIGNMENT_2_TORCH_DEVICE=mps requested but MPS is unavailable; falling back to auto selection.")
        else:
            return requested_device

    ram_gb = total_system_ram_gb()
    if (
        platform.system() == "Darwin"
        and platform.machine() == "arm64"
        and mps_is_available()
        and ram_gb >= min_mps_ram_gb
    ):
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"

MIN_MPS_RAM_GB = int(os.getenv("ASSIGNMENT_2_MIN_MPS_RAM_GB", "64"))
LOCAL_TORCH_DEVICE = choose_local_torch_device(MIN_MPS_RAM_GB)
print(f"Local embedding/reranker torch device: {LOCAL_TORCH_DEVICE} (system RAM: {total_system_ram_gb():.1f} GB)")

# API-bound loops run concurrently. Tune down if the provider rate-limits you.
API_GENERATION_WORKERS = int(os.getenv("ASSIGNMENT_2_API_GENERATION_WORKERS", "100"))
API_JUDGE_WORKERS = int(os.getenv("ASSIGNMENT_2_API_JUDGE_WORKERS", "100"))
API_FAITHFULNESS_WORKERS = int(os.getenv("ASSIGNMENT_2_API_FAITHFULNESS_WORKERS", "4"))

def run_parallel_jobs(items, worker, desc: str, max_workers: int, checkpoint_path=None):
    items = list(items)
    if not items:
        return []
    max_workers = max(1, min(int(max_workers), len(items)))
    results = [None] * len(items)
    completed_rows = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(worker, item): index for index, item in enumerate(items)}
        for future in tqdm(as_completed(futures), total=len(futures), desc=f"{desc} parallel x{max_workers}"):
            index = futures[future]
            result = future.result()
            results[index] = result
            completed_rows.append(result)
            if checkpoint_path is not None:
                checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
                checkpoint_path.write_text(json.dumps(completed_rows, indent=2, ensure_ascii=False))
    return results

# Task 3
VECTORSTORE_DIR = DATA_DIR / "vectorstore" / "financebench_bge_small_v1_5"
VECTORSTORE_MANIFEST_PATH = VECTORSTORE_DIR / "manifest.json"
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
REBUILD_VECTORSTORE = False
TASK3_SAMPLE_IDS = [
    "financebench_id_00005",  # Corning working capital; company appears in the question.
    "financebench_id_00283",  # Pfizer / Upjohn future payment; specific filing fact.
    "financebench_id_00288",  # Best Buy cash question; company is not named in the question text.
]
TASK3_TOP_K = 5
STOPWORDS = {
    "the", "and", "for", "from", "with", "that", "this", "were", "was", "are", "our",
    "its", "into", "has", "have", "had", "not", "but", "you", "your", "their", "which",
    "between", "based", "please", "state", "explain", "company", "data", "year",
}

# Task 4
RAG_VECTORSTORE_DIR = DATA_DIR / "vectorstore" / "financebench_bge_small_v1_5"
RAG_EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
RAG_GENERATION_MODEL = "meta-llama/Llama-3.3-70B-Instruct"
RAG_SYSTEM_PROMPT = """Role: You are a financial question-answering assistant for FinanceBench.

Grounding rules:
- Use only the retrieved FinanceBench context in the user message.
- If the retrieved context does not contain enough evidence, say exactly: The context does not contain enough information.
- Do not use outside knowledge, memory, or assumptions to fill missing filing data.
- Prefer chunks whose company, filing name, fiscal period, and date match the question.
- Ignore chunks from mismatched companies or periods unless the question explicitly asks for a comparison.
- Preserve units, fiscal periods, dates, signs, and directionality for numeric answers.
- Cite every factual sentence with [doc_name, page N].

Numeric answer format:
- Answer: <value and short conclusion>
- Unit: <unit or Not applicable>
- Period/date: <period/date or Not specified>
- Formula: <formula used, or Not applicable>
- Evidence: <citations>

Answer style:
- Be concise.
- Include only the final answer and citations; do not describe hidden reasoning."""
RAG_FEW_SHOT_MESSAGES = [
    {
        "role": "user",
        "content": """Question:
Did ExampleCo revenue increase in FY2023?

Retrieved FinanceBench context:
--- Chunk 1 ---
doc_name: EXAMPLECO_2023_10K
company: ExampleCo
doc_period: 2023
page_number: 42
content:
Revenue was $120 million in fiscal 2023 compared with $100 million in fiscal 2022.

Instructions:
- Answer using only the retrieved context above.
- If the answer is not present, say exactly: The context does not contain enough information.
- Cite each factual sentence with [doc_name, page N].""",
    },
    {
        "role": "assistant",
        "content": "Answer: Yes, ExampleCo revenue increased from $100 million in FY2022 to $120 million in FY2023. Unit: USD millions. Period/date: FY2023 compared with FY2022. Formula: Not applicable. Evidence: [EXAMPLECO_2023_10K, page 42].",
    },
    {
        "role": "user",
        "content": """Question:
By how much did ExampleCo operating income change from FY2022 to FY2023?

Retrieved FinanceBench context:
--- Chunk 1 ---
doc_name: EXAMPLECO_2023_10K
company: ExampleCo
doc_period: 2023
page_number: 58
content:
Operating income was $45 million in fiscal 2023 and $30 million in fiscal 2022.

Instructions:
- Answer using only the retrieved context above.
- If the answer is not present, say exactly: The context does not contain enough information.
- Cite each factual sentence with [doc_name, page N].""",
    },
    {
        "role": "assistant",
        "content": "Answer: Operating income increased by $15 million. Unit: USD millions. Period/date: FY2023 compared with FY2022. Formula: $45 million - $30 million = $15 million. Evidence: [EXAMPLECO_2023_10K, page 58].",
    },
    {
        "role": "user",
        "content": """Question:
What was ExampleCo capital expenditure in FY2023?

Retrieved FinanceBench context:
--- Chunk 1 ---
doc_name: EXAMPLECO_2023_10K
company: ExampleCo
doc_period: 2023
page_number: 12
content:
ExampleCo describes its operating segments and revenue by region.

Instructions:
- Answer using only the retrieved context above.
- If the answer is not present, say exactly: The context does not contain enough information.
- Cite each factual sentence with [doc_name, page N].""",
    },
    {
        "role": "assistant",
        "content": "The context does not contain enough information.",
    },
]

# Task 5
TASK5_RAG_K = 5
TASK5_RAW_PATH = ARTIFACT_DIR / "task5_rag_answers_raw.json"
TASK5_XLSX_PATH = OUTPUT_DIR / "assignment_2_run_and_compare.xlsx"
FORCE_TASK5_REGEN = False

TASK5_REQUIRED_COLUMNS = [
    "financebench_id",
    "question_type",
    "question",
    "naive_answer",
    "RAG_answer",
    "ground_truth",
]

# Task 6
TASK6_DATASET_PATH = FILTERED_DATASET_PATH
TASK6_RAG_ALL_RAW_PATH = ARTIFACT_DIR / "task6_rag_all_answers_raw.json"
TASK6_CORRECTNESS_RAW_PATH = ARTIFACT_DIR / "task6_correctness_raw.json"
TASK6_SUPPORT_RAW_PATH = ARTIFACT_DIR / "task6_support_raw.json"
TASK6_FAITHFULNESS_RAW_PATH = ARTIFACT_DIR / "task6_faithfulness_raw.json"
TASK6_XLSX_PATH = OUTPUT_DIR / "assignment_2_evaluation.xlsx"
TASK6_RAG_K = 5
TASK6_PAGE_HIT_K_VALUES = [1, 3, 5]
TASK6_FAITHFULNESS_LIMIT = 20
TASK6_RAGAS_MAX_TOKENS = 8192
TASK6_RAGAS_SCORE_TIMEOUT_SECONDS = int(os.getenv("ASSIGNMENT_2_RAGAS_SCORE_TIMEOUT_SECONDS", "45"))
TASK6_RAGAS_MAX_RETRIES = int(os.getenv("ASSIGNMENT_2_RAGAS_MAX_RETRIES", "2"))
TASK6_FAITHFULNESS_CONTEXT_MAX_CHARS = 1200
TASK6_SUPPORT_CONTEXT_MAX_CHARS = 1600
TASK6_JUDGE_MODEL = os.getenv("NEBIUS_JUDGE_MODEL", "deepseek-ai/DeepSeek-V3.2")
TASK6_JUDGE_RUN_VERSION = "deepseek_v3_2_json_v1"
TASK6_SUPPORT_RUN_VERSION = "deepseek_v3_2_support_json_v1"
FORCE_TASK6_RAG_REGEN = False
FORCE_TASK6_CORRECTNESS_REGEN = False
FORCE_TASK6_SUPPORT_REGEN = False
FORCE_TASK6_FAITHFULNESS_REGEN = False

# Task 7
TASK7_OUTPUT_XLSX_PATH = OUTPUT_DIR / "assignment_2_improvement_cycles.xlsx"
TASK7_ARTIFACT_DIR = ARTIFACT_DIR
FORCE_TASK7_REGEN = False
TASK7_FAITHFULNESS_LIMIT = TASK6_FAITHFULNESS_LIMIT
TASK7_BASELINE_PAGE_HIT_K_VALUES = TASK6_PAGE_HIT_K_VALUES
TASK7_BASELINE_NAME = "task6_baseline"
TASK7_STRICT_SYSTEM_PROMPT = """Role: You are a careful financial QA assistant for FinanceBench.

Evidence rules:
- Use only the retrieved FinanceBench context in the user message.
- If the retrieved context does not contain enough evidence, say exactly: The context does not contain enough information.
- Prefer chunks whose company, filing name, fiscal period, and date match the question.
- Ignore chunks from mismatched companies or periods unless the question explicitly asks for a comparison.
- Do not blend facts from different documents unless the question explicitly asks for a comparison.
- When a question asks for a number, preserve the relevant units, fiscal periods, dates, signs, and directionality from the context.
- Cite every factual sentence with [doc_name, page N].

Numeric answer format:
- Answer: <value and short conclusion>
- Unit: <unit or Not applicable>
- Period/date: <period/date or Not specified>
- Formula: <formula used, or Not applicable>
- Evidence: <citations>

Answer style:
- Be concise.
- Include only the final answer and citations; do not describe hidden reasoning."""
TASK7_EXPERIMENTS = [
    {
        "experiment": "prompt_no_few_shot",
        "change": "A/B prompt ablation: reused Task 6 top-5 chunks and base system prompt, but removed few-shot examples.",
        "hypothesis": "Removing few-shot examples should reduce answer-format consistency and may reduce correctness; retrieval metrics should stay unchanged.",
        "retrieval_mode": "baseline_chunks",
        "system_prompt": RAG_SYSTEM_PROMPT,
        "few_shot_messages": [],
        "page_hit_k_values": [1, 3, 4, 5],
        "generation_model": RAG_GENERATION_MODEL,
        "final_k": TASK6_RAG_K,
    },
    {
        "experiment": "strict_prompt",
        "change": "Changed only the generation system prompt; reused the Task 6 top-5 retrieved chunks.",
        "hypothesis": "A stricter evidence-and-citation prompt should improve faithfulness and slightly improve correctness; retrieval metrics should stay unchanged.",
        "retrieval_mode": "baseline_chunks",
        "system_prompt": TASK7_STRICT_SYSTEM_PROMPT,
        "page_hit_k_values": [1, 3, 4, 5],
        "generation_model": RAG_GENERATION_MODEL,
        "final_k": TASK6_RAG_K,
    },
    {
        "experiment": "top_k_8",
        "change": "Changed only the number of FAISS chunks sent to the generator from k=5 to k=8.",
        "hypothesis": "More chunks should improve page-hit@8 and may improve correctness, but extra context may distract the generator.",
        "retrieval_mode": "faiss",
        "retrieval_k": 8,
        "page_hit_k_values": [1, 3, 4, 5, 8],
        "system_prompt": RAG_SYSTEM_PROMPT,
        "generation_model": RAG_GENERATION_MODEL,
        "final_k": 8,
    },
    {
        "experiment": "bge_reranker_top4",
        "change": "Added BAAI/bge-reranker-base: retrieve top-20 with FAISS, then rerank to top-4 for generation.",
        "hypothesis": "Reranking should improve top-ranked evidence quality by selecting the best four chunks from the initial FAISS top-20 candidates.",
        "retrieval_mode": "reranker",
        "candidate_k": 20,
        "final_k": 4,
        "reranker_model": "BAAI/bge-reranker-base",
        "page_hit_k_values": [1, 3, 4],
        "system_prompt": RAG_SYSTEM_PROMPT,
        "generation_model": RAG_GENERATION_MODEL,
    },
]

# Bonus
BONUS_DATASET_PATH = FILTERED_DATASET_PATH
BONUS_PDF_DIR = PDF_DIR
BONUS_ARTIFACT_DIR = ARTIFACT_DIR
BONUS_XLSX_PATH = OUTPUT_DIR / "assignment_2_bonus_multiscale_chunking.xlsx"
BONUS_EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
BONUS_CHUNK_OVERLAP = 150
BONUS_CHUNK_SIZES = [500, 1000, 2000]
BONUS_INDEX_DIRS = {
    500: DATA_DIR / "vectorstore" / "financebench_bge_small_v1_5_chunk500",
    1000: DATA_DIR / "vectorstore" / "financebench_bge_small_v1_5",
    2000: DATA_DIR / "vectorstore" / "financebench_bge_small_v1_5_chunk2000",
}
BONUS_REBUILD_INDICES = False


Local embedding/reranker torch device: mps (system RAM: 128.0 GB)


## Dataset Preparation

Load FinanceBench, drop `metrics-generated`, repair `doc_link` values using the FinanceBench PDF repository, and normalize evidence page numbers for later retrieval evaluation.


In [5]:
try:
    df_raw = pd.read_json(DATASET_URL, lines=True)
    dataset_source = DATASET_URL
except Exception as exc:
    print(f"hf:// load failed ({type(exc).__name__}): {exc}")
    df_raw = pd.read_json(DATASET_FALLBACK_URL, lines=True)
    dataset_source = DATASET_FALLBACK_URL

print(f"Loaded dataset from: {dataset_source}")
print(f"Raw dataset shape: {df_raw.shape}")
print("Columns:")
print(df_raw.columns.tolist())
print(df_raw.head(3))


Loaded dataset from: hf://datasets/PatronusAI/financebench/financebench_merged.jsonl
Raw dataset shape: (150, 15)
Columns:
['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link']
         financebench_id company     doc_name      question_type  \
0  financebench_id_03029      3M  3M_2018_10K  metrics-generated   
1  financebench_id_04672      3M  3M_2018_10K  metrics-generated   
2  financebench_id_00499      3M  3M_2022_10K    domain-relevant   

                                 question_reasoning domain_question_num  \
0                            Information extraction                None   
1                            Information extraction                None   
2  Logical reasoning (based on numerical reasoning)                dg06   

                                            question  \
0  What is the

In [6]:
def ensure_evidence_list(value):
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        return [value]
    if pd.isna(value):
        return []
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        for parser in (json.loads, ast.literal_eval):
            try:
                parsed = parser(text)
                if isinstance(parsed, list):
                    return parsed
                if isinstance(parsed, dict):
                    return [parsed]
            except Exception:
                pass
    return []

def extract_evidence_page_nums(items):
    page_nums = []
    for item in items:
        if not isinstance(item, dict):
            continue
        raw_page = item.get("page_num", item.get("evidence_page_num", item.get("evidence__page_num")))
        try:
            if raw_page is not None and str(raw_page).strip() != "":
                page_nums.append(int(raw_page))
        except (TypeError, ValueError):
            continue
    return sorted(set(page_nums))

def extract_evidence_texts(items):
    texts = []
    for item in items:
        if not isinstance(item, dict):
            continue
        raw_text = item.get("evidence") or item.get("text") or item.get("evidence_text")
        if raw_text:
            texts.append(str(raw_text).strip())
    return texts

def normalize_doc_filename(doc_name, doc_link):
    candidates = []
    if isinstance(doc_link, str) and doc_link.strip():
        link_tail = doc_link.rstrip("/").split("/")[-1]
        link_tail = link_tail.split("?")[0].split("#")[0]
        if link_tail:
            candidates.append(link_tail)
    if isinstance(doc_name, str) and doc_name.strip():
        name = doc_name.strip()
        candidates.append(name if name.lower().endswith(".pdf") else f"{name}.pdf")
    for candidate in candidates:
        if candidate.lower().endswith(".pdf"):
            return candidate
    return None

def build_repo_links(filename):
    if not filename:
        return None, None
    return (
        f"{PDF_REPO_BASE}/{filename}",
        f"{PDF_RAW_BASE}/{filename}",
    )


In [7]:
df = df_raw.copy()
df["question_type"] = df["question_type"].astype(str).str.strip().str.lower()

before_counts = df["question_type"].value_counts(dropna=False).sort_index()

df["evidence_items"] = df["evidence"].apply(ensure_evidence_list)
df["evidence_page_nums"] = df["evidence_items"].apply(extract_evidence_page_nums)
df["evidence_texts"] = df["evidence_items"].apply(extract_evidence_texts)

df["doc_filename"] = df.apply(lambda row: normalize_doc_filename(row.get("doc_name"), row.get("doc_link")), axis=1)
repo_links = df["doc_filename"].apply(build_repo_links)
df["doc_link_repo"] = repo_links.apply(lambda pair: pair[0])
df["doc_link_raw"] = repo_links.apply(lambda pair: pair[1])
df["doc_link_original"] = df["doc_link"]
df["doc_link"] = df["doc_link_repo"].where(df["doc_link_repo"].notna(), df["doc_link"])

df = df[df["question_type"].isin(KEEP_QUESTION_TYPES)].copy()
df["financebench_id_numeric"] = pd.to_numeric(df["financebench_id"], errors="coerce")
df = df.sort_values(["financebench_id_numeric", "financebench_id"], kind="stable").drop(columns=["financebench_id_numeric"]).reset_index(drop=True)

after_counts = df["question_type"].value_counts(dropna=False).sort_index()
preview_cols = [
    col
    for col in [
        "financebench_id",
        "question_type",
        "company",
        "doc_name",
        "doc_filename",
        "doc_link",
        "doc_link_raw",
        "evidence_page_nums",
        "question",
    ]
    if col in df.columns
]

print("Question-type counts before filtering:")
print(before_counts.to_frame("count").to_string())

print("Question-type counts after filtering:")
print(after_counts.to_frame("count").to_string())

print(f"Filtered dataset shape: {df.shape}")
print(f"Rows missing doc filename: {int(df['doc_filename'].isna().sum())}")
print(df[preview_cols].head(10))


Question-type counts before filtering:
                   count
question_type           
domain-relevant       50
metrics-generated     50
novel-generated       50
Question-type counts after filtering:
                 count
question_type         
domain-relevant     50
novel-generated     50
Filtered dataset shape: (100, 22)
Rows missing doc filename: 0
         financebench_id    question_type               company  \
0  financebench_id_00005  domain-relevant               Corning   
1  financebench_id_00070  domain-relevant  American Water Works   
2  financebench_id_00080  domain-relevant                Paypal   
3  financebench_id_00206  domain-relevant              JPMorgan   
4  financebench_id_00215  domain-relevant               Verizon   
5  financebench_id_00216  domain-relevant               Verizon   
6  financebench_id_00222  domain-relevant                   AMD   
7  financebench_id_00283  novel-generated                Pfizer   
8  financebench_id_00288  novel-generate

In [8]:
clean_dataset_path = FILTERED_DATASET_PATH
clean_dataset_path.parent.mkdir(exist_ok=True)
df.to_json(clean_dataset_path, orient="records", lines=True, force_ascii=False)
print(f"Saved cleaned dataset to {clean_dataset_path}")


Saved cleaned dataset to data/financebench_filtered.jsonl


## Task 1 - Naive Generation

Use `meta-llama/Llama-3.3-70B-Instruct` through the Nebius Token Factory OpenAI-compatible endpoint to answer the first five `domain-relevant` and first five `novel-generated` questions after sorting by `financebench_id`.

This section intentionally does **not** retrieve evidence or document text. Each API call sends a model-specific system message plus the question text as the user message. The API key is read from `NEBIUS_API_KEY` or requested interactively with `getpass`, and is not stored in the notebook.

In [9]:
df_task1 = pd.read_json(FILTERED_DATASET_PATH, lines=True)

task1_questions = (
    df_task1.sort_values("financebench_id", kind="stable")
    .groupby("question_type", group_keys=False)
    .head(5)
    .sort_values(["question_type", "financebench_id"], kind="stable")
    .reset_index(drop=True)
)

display(task1_questions[["financebench_id", "question_type", "doc_name", "question", "answer"]])


,financebench_id,question_type,doc_name,question,answer
0,financebench_id_00005,domain-relevant,CORNING_2022_10K,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...
1,financebench_id_00070,domain-relevant,AMERICANWATERWORKS_2022_10K,Does American Water Works have positive workin...,"No, American Water Works had negative working ..."
2,financebench_id_00080,domain-relevant,PAYPAL_2022_10K,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...
3,financebench_id_00206,domain-relevant,JPMORGAN_2022_10K,Are JPM's gross margins historically consisten...,"Since JPM is a financial institution, gross ma..."
4,financebench_id_00215,domain-relevant,VERIZON_2022_10K,Is Verizon a capital intensive business based ...,Yes. Verizon's capital intensity ratio was app...
5,financebench_id_00283,novel-generated,Pfizer_2023Q2_10Q,How much does Pfizer expect to pay to spin off...,77.78
6,financebench_id_00288,novel-generated,BESTBUY_2024Q2_10Q,Was there any drop in Cash & Cash equivalents ...,"Yes, there was a decline of ~42% between FY202..."
7,financebench_id_00299,novel-generated,JPMORGAN_2021Q1_10Q,Which of JPM's business segments had the lowes...,Corporate. Its net revenue was -$473 million.
8,financebench_id_00302,novel-generated,PFIZER_2021_10K,Did Pfizer grow its PPNE between FY20 and FY21?,"Yes, change in PPNE was positive year over year"
9,financebench_id_00382,novel-generated,MGMRESORTS_2022Q4_EARNINGS,Which region had the Highest EBITDAR Contribut...,Las Vegas resorts contributed ~90% of company ...


In [10]:
def get_nebius_client() -> OpenAI:
    api_key = os.getenv("NEBIUS_API_KEY") or getpass("Nebius API key: ")
    if not api_key:
        raise ValueError("Set NEBIUS_API_KEY or enter a Nebius API key to run generation.")
    ensure_api_ca_bundle()
    return OpenAI(base_url=NEBIUS_BASE_URL, api_key=api_key, timeout=60.0, max_retries=API_MAX_RETRIES)


def answer_naively(client: OpenAI, question: str) -> str:
    def _request():
        response = client.chat.completions.create(
            model=NAIVE_GENERATION_MODEL,
            messages=[
                {"role": "system", "content": NAIVE_SYSTEM_PROMPT},
                {"role": "user", "content": question},
            ],
            temperature=0,
            max_tokens=700,
        )
        return response.choices[0].message.content.strip()

    return call_with_retries(_request, "Naive generation")


def load_task1_cached_rows(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = json.loads(path.read_text())
    if not isinstance(rows, list):
        raise ValueError(f"Expected a JSON list in {path}, found {type(rows).__name__}")
    return rows


def has_complete_task1_cache(rows: list[dict], expected_ids: list[str]) -> bool:
    if len(rows) != len(expected_ids):
        return False
    by_id = {row.get("financebench_id"): row for row in rows if isinstance(row, dict)}
    if set(by_id) != set(expected_ids):
        return False
    return all(
        isinstance(by_id[financebench_id].get("naive_answer"), str)
        and isinstance(by_id[financebench_id].get("ground_truth"), str)
        for financebench_id in expected_ids
    )


task1_expected_ids = task1_questions["financebench_id"].tolist()
task1_order = {financebench_id: index for index, financebench_id in enumerate(task1_expected_ids)}

if TASK1_RAW_PATH.exists() and not FORCE_TASK1_REGEN:
    cached_task1_rows = load_task1_cached_rows(TASK1_RAW_PATH)
    if has_complete_task1_cache(cached_task1_rows, task1_expected_ids):
        task1_raw_rows = sorted(cached_task1_rows, key=lambda row: task1_order[row["financebench_id"]])
        print(f"Loaded cached raw naive generations from {TASK1_RAW_PATH}")
    else:
        print(f"Cached Task 1 artifact at {TASK1_RAW_PATH} is incomplete; regenerating it.")
        client = get_nebius_client()

        def task1_generate_row(item: tuple[int, pd.Series]) -> dict:
            _, row = item
            naive_answer = answer_naively(client, row["question"])
            return {
                "financebench_id": row["financebench_id"],
                "question_type": row["question_type"],
                "question": row["question"],
                "naive_answer": naive_answer,
                "ground_truth": row["answer"],
                "doc_name": row.get("doc_name"),
                "evidence_page_nums": row.get("evidence_page_nums"),
            }

        task1_raw_rows = run_parallel_jobs(
            task1_questions.iterrows(),
            task1_generate_row,
            desc="Task 1 naive generation",
            max_workers=API_GENERATION_WORKERS,
            checkpoint_path=TASK1_RAW_PATH,
        )
        TASK1_RAW_PATH.write_text(json.dumps(task1_raw_rows, indent=2, ensure_ascii=False))
        print(f"Saved raw naive generations to {TASK1_RAW_PATH}")
else:
    client = get_nebius_client()

    def task1_generate_row(item: tuple[int, pd.Series]) -> dict:
        _, row = item
        naive_answer = answer_naively(client, row["question"])
        return {
            "financebench_id": row["financebench_id"],
            "question_type": row["question_type"],
            "question": row["question"],
            "naive_answer": naive_answer,
            "ground_truth": row["answer"],
            "doc_name": row.get("doc_name"),
            "evidence_page_nums": row.get("evidence_page_nums"),
        }

    task1_raw_rows = run_parallel_jobs(
        task1_questions.iterrows(),
        task1_generate_row,
        desc="Task 1 naive generation",
        max_workers=API_GENERATION_WORKERS,
        checkpoint_path=TASK1_RAW_PATH,
    )
    TASK1_RAW_PATH.write_text(json.dumps(task1_raw_rows, indent=2, ensure_ascii=False))
    print(f"Saved raw naive generations to {TASK1_RAW_PATH}")

naive_generation_df = pd.DataFrame(task1_raw_rows)
display(naive_generation_df[["financebench_id", "question_type", "question", "naive_answer", "ground_truth"]])


Loaded cached raw naive generations from rag_faiss/artifacts/task1_naive_generation_raw.json


,financebench_id,question_type,question,naive_answer,ground_truth
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,The answer cannot be determined from the quest...,Yes. Corning had a positive working capital am...
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,The answer cannot be determined from the quest...,"No, American Water Works had negative working ..."
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,The answer cannot be determined from the quest...,Yes. Paypal has a positive working capital of ...
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,Gross margins are not a relevant metric for a ...,"Since JPM is a financial institution, gross ma..."
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,The answer cannot be determined from the quest...,Yes. Verizon's capital intensity ratio was app...
5,financebench_id_00283,novel-generated,How much does Pfizer expect to pay to spin off...,The answer cannot be determined from the quest...,77.78
6,financebench_id_00288,novel-generated,Was there any drop in Cash & Cash equivalents ...,The question does not provide the Cash & Cash ...,"Yes, there was a decline of ~42% between FY202..."
7,financebench_id_00299,novel-generated,Which of JPM's business segments had the lowes...,The answer cannot be determined from the quest...,Corporate. Its net revenue was -$473 million.
8,financebench_id_00302,novel-generated,Did Pfizer grow its PPNE between FY20 and FY21?,The question does not provide the PPNE (Pfizer...,"Yes, change in PPNE was positive year over year"
9,financebench_id_00382,novel-generated,Which region had the Highest EBITDAR Contribut...,The answer cannot be determined from the quest...,Las Vegas resorts contributed ~90% of company ...


In [11]:
# Manual judgments after comparing each model answer with the FinanceBench gold answer.
# "Refused" includes answers that say the result cannot be determined from the question alone.
task1_verdicts = {
    "financebench_id_00005": "refused",
    "financebench_id_00070": "refused",
    "financebench_id_00080": "refused",
    "financebench_id_00206": "correct",
    "financebench_id_00215": "refused",
    "financebench_id_00283": "refused",
    "financebench_id_00288": "refused",
    "financebench_id_00299": "refused",
    "financebench_id_00302": "refused",
    "financebench_id_00382": "refused",
}

task1_judgment_notes = {
    "financebench_id_00005": "The model said the working-capital answer cannot be determined without Corning's FY2022 balance-sheet data.",
    "financebench_id_00070": "The model said the working-capital answer cannot be determined without American Water Works FY2022 data.",
    "financebench_id_00080": "The model said the working-capital answer cannot be determined without PayPal's FY2022 current assets and liabilities.",
    "financebench_id_00206": "The model correctly identified that gross margin is not a relevant metric for a bank such as JPMorgan.",
    "financebench_id_00215": "The model declined to answer without FY2022 capital-intensity data, while the gold answer says Verizon is capital intensive.",
    "financebench_id_00283": "The model declined to answer without filing data; it did not provide the gold $77.78M amount.",
    "financebench_id_00288": "The model declined to answer without the cash values for FY2023 and Q2 FY2024.",
    "financebench_id_00299": "The model declined to answer without JPM segment revenue data.",
    "financebench_id_00302": "The model declined to answer and did not identify whether Pfizer PPNE grew from FY20 to FY21.",
    "financebench_id_00382": "The model declined to answer without MGM EBITDAR contribution data.",
}

assignment_2_naive_generation = naive_generation_df.copy()
assignment_2_naive_generation["verdict"] = assignment_2_naive_generation["financebench_id"].map(task1_verdicts)
assignment_2_naive_generation["judgment_note"] = assignment_2_naive_generation["financebench_id"].map(task1_judgment_notes)

required_task1_columns = [
    "financebench_id",
    "question_type",
    "question",
    "naive_answer",
    "ground_truth",
    "verdict",
]

TASK1_XLSX_PATH.parent.mkdir(exist_ok=True)
assignment_2_naive_generation[required_task1_columns].to_excel(TASK1_XLSX_PATH, index=False)
print(f"Saved Task 1 deliverable to {TASK1_XLSX_PATH}")

display(assignment_2_naive_generation[required_task1_columns + ["judgment_note"]])
display(
    assignment_2_naive_generation
    .groupby(["question_type", "verdict"])
    .size()
    .rename("count")
    .reset_index()
)


Saved Task 1 deliverable to rag_faiss/outputs/assignment_2_naive_generation.xlsx


,financebench_id,question_type,question,naive_answer,ground_truth,verdict,judgment_note
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,The answer cannot be determined from the quest...,Yes. Corning had a positive working capital am...,refused,The model said the working-capital answer cann...
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,The answer cannot be determined from the quest...,"No, American Water Works had negative working ...",refused,The model said the working-capital answer cann...
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,The answer cannot be determined from the quest...,Yes. Paypal has a positive working capital of ...,refused,The model said the working-capital answer cann...
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,Gross margins are not a relevant metric for a ...,"Since JPM is a financial institution, gross ma...",correct,The model correctly identified that gross marg...
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,The answer cannot be determined from the quest...,Yes. Verizon's capital intensity ratio was app...,refused,The model declined to answer without FY2022 ca...
5,financebench_id_00283,novel-generated,How much does Pfizer expect to pay to spin off...,The answer cannot be determined from the quest...,77.78,refused,The model declined to answer without filing da...
6,financebench_id_00288,novel-generated,Was there any drop in Cash & Cash equivalents ...,The question does not provide the Cash & Cash ...,"Yes, there was a decline of ~42% between FY202...",refused,The model declined to answer without the cash ...
7,financebench_id_00299,novel-generated,Which of JPM's business segments had the lowes...,The answer cannot be determined from the quest...,Corporate. Its net revenue was -$473 million.,refused,The model declined to answer without JPM segme...
8,financebench_id_00302,novel-generated,Did Pfizer grow its PPNE between FY20 and FY21?,The question does not provide the PPNE (Pfizer...,"Yes, change in PPNE was positive year over year",refused,The model declined to answer and did not ident...
9,financebench_id_00382,novel-generated,Which region had the Highest EBITDAR Contribut...,The answer cannot be determined from the quest...,Las Vegas resorts contributed ~90% of company ...,refused,The model declined to answer without MGM EBITD...


,question_type,verdict,count
0,domain-relevant,correct,1
1,domain-relevant,refused,4
2,novel-generated,refused,5


### Task 1 Discussion

**Refusals / requests for more information.** The naive model refused or asked for more information on 9 of the 10 questions. It usually said the answer could not be determined from the question alone because the prompt did not include the relevant filing values, segment data, or balance-sheet numbers. This was especially visible for the working-capital questions, Pfizer/Upjohn, Best Buy cash, JPM segment revenue, Pfizer PPNE, and MGM EBITDAR.

**Confident answers.** The only clearly correct substantive answer was `financebench_id_00206`: the model identified that gross margin is not a relevant metric for a bank such as JPMorgan. The Verizon answer was not counted as correct because the model declined to determine capital intensity instead of answering the gold yes/no result. The other questions did not contain enough information in the user prompt, and the model mostly avoided hallucinating specific filing numbers.

**Patterns by question type.** The `domain-relevant` set had one correct conceptual answer and four refusals. The `novel-generated` set had five refusals. The novel-generated questions were more filing-specific and often impossible to answer from the standalone question, but even the domain-relevant working-capital questions required exact FY2022 statement data that was not supplied to the naive model.


## Task 2 - RAG Reminder

**Indexing: documents -> chunk + embed -> vector store.** Indexing converts the source documents into searchable evidence. In this assignment, that means loading the relevant FinanceBench filings, splitting each page into chunks, embedding each chunk, and storing the vectors with metadata such as `doc_name`, `company`, `doc_period`, and `page_number`. This is what gives the later pipeline a knowledge base instead of relying only on the model's memory. It can fail if the wrong PDFs are indexed, if PDF extraction corrupts tables, if chunks split a calculation across two pieces, if page metadata is off by one, if the embedding model does not represent financial wording well, or if the vector store becomes stale after document or preprocessing changes. This usually happens offline once for a fixed corpus, chunking strategy, and embedding model, and is rerun only when one of those inputs changes.

**Retrieval: user query -> retrieved context.** Retrieval embeds the user question and searches the vector store for the chunks that are most likely to contain the answer. Its job is to narrow thousands of pages down to a small evidence set that can fit in the generation prompt. It can fail when the query is under-specified, such as asking whether cash decreased without naming the company; when equivalent terms do not match well, such as `PPNE` versus `PP&E`; when the top-k value is too small and misses the evidence page; when top-k is too large and adds distracting chunks; or when retrieval finds the right metric in the wrong filing period or company because metadata filters were not used. Retrieval happens per query, because each user question needs its own search results.

**Generation: query + retrieved chunks -> final answer.** Generation uses the retrieved context and the original question to produce the final response. In a good RAG pipeline, the model should answer only from the supplied chunks, perform any needed synthesis or arithmetic, and cite the relevant document/page metadata. It can fail if the retrieved context is incomplete and the model guesses anyway, if it performs financial calculations incorrectly, if it latches onto a distracting chunk, if it ignores instructions to say that the answer is not present, or if it gives a correct-looking answer with an unsupported citation. Generation happens per query. The prompt template can be designed offline, but the actual model call and final answer are produced every time a query is asked.

## Task 3 - Embed Documents

This section builds the FinanceBench knowledge base for later RAG tasks. It downloads only the PDFs referenced by the filtered dataset, loads them one page at a time with `PyPDFLoader`, attaches standardized page metadata, chunks pages with `RecursiveCharacterTextSplitter`, embeds chunks with `BAAI/bge-small-en-v1.5`, and saves the FAISS index locally for reuse.


In [12]:
PDF_DIR.mkdir(parents=True, exist_ok=True)
VECTORSTORE_DIR.parent.mkdir(parents=True, exist_ok=True)

df_task3 = pd.read_json(FILTERED_DATASET_PATH, lines=True)

def repo_pdf_filename(doc_name: str) -> str:
    return f"{doc_name}.pdf"

def raw_pdf_url(filename: str) -> str:
    return f"{PDF_RAW_BASE}/{urllib.parse.quote(filename)}"

relevant_docs = (
    df_task3[["doc_name", "company", "doc_period", "doc_filename", "doc_link_raw"]]
    .drop_duplicates(subset=["doc_name"])
    .sort_values("doc_name", kind="stable")
    .reset_index(drop=True)
)
relevant_docs["repo_pdf_filename"] = relevant_docs["doc_name"].map(repo_pdf_filename)
relevant_docs["pdf_url"] = relevant_docs["repo_pdf_filename"].map(raw_pdf_url)
relevant_docs["local_pdf_path"] = relevant_docs["repo_pdf_filename"].map(lambda name: str(PDF_DIR / name))

print(f"Filtered dataset rows: {len(df_task3):,}")
print(f"Relevant PDFs: {len(relevant_docs):,}")
display(relevant_docs[["doc_name", "company", "doc_period", "repo_pdf_filename"]].head(10))

def download_pdf(row: pd.Series) -> Path:
    """Download one FinanceBench PDF if it is not already present locally."""
    target_path = PDF_DIR / row["repo_pdf_filename"]
    if target_path.exists() and target_path.stat().st_size > 0:
        return target_path

    request = urllib.request.Request(
        row["pdf_url"],
        headers={"User-Agent": "assignment_2-financebench-rag/1.0"},
    )
    tmp_path = target_path.with_suffix(target_path.suffix + ".part")
    with urllib.request.urlopen(request, timeout=120) as response, tmp_path.open("wb") as output_file:
        shutil.copyfileobj(response, output_file)
    tmp_path.replace(target_path)
    return target_path

pdf_paths = []
for _, row in tqdm(relevant_docs.iterrows(), total=len(relevant_docs), desc="Downloading PDFs"):
    pdf_paths.append(download_pdf(row))

missing_or_empty = [path for path in pdf_paths if not path.exists() or path.stat().st_size == 0]
if missing_or_empty:
    raise RuntimeError(f"Missing or empty PDFs: {missing_or_empty[:5]}")

print(f"PDFs available in {PDF_DIR}: {len(pdf_paths):,}")

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": LOCAL_TORCH_DEVICE},
    encode_kwargs={"normalize_embeddings": True},
)

index_exists = (VECTORSTORE_DIR / "index.faiss").exists() and (VECTORSTORE_DIR / "index.pkl").exists()

if index_exists and not REBUILD_VECTORSTORE:
    vectorstore = FAISS.load_local(
        str(VECTORSTORE_DIR),
        embedding_model,
        allow_dangerous_deserialization=True,
    )
    print(f"Loaded existing FAISS index from {VECTORSTORE_DIR}")
    if VECTORSTORE_MANIFEST_PATH.exists():
        manifest = json.loads(VECTORSTORE_MANIFEST_PATH.read_text())
        print(json.dumps(manifest, indent=2))
else:
    pages = []
    load_errors = []

    for _, row in tqdm(relevant_docs.iterrows(), total=len(relevant_docs), desc="Loading PDF pages"):
        pdf_path = PDF_DIR / row["repo_pdf_filename"]
        try:
            loaded_pages = PyPDFLoader(str(pdf_path)).load()
        except Exception as exc:
            load_errors.append({"doc_name": row["doc_name"], "pdf_path": str(pdf_path), "error": repr(exc)})
            continue

        for fallback_page_number, page in enumerate(loaded_pages):
            page_number = int(page.metadata.get("page", fallback_page_number))
            page.metadata.update(
                {
                    "doc_name": row["doc_name"],
                    "company": row["company"],
                    "doc_period": int(row["doc_period"]),
                    "page_number": page_number,
                }
            )
            page.metadata.pop("page", None)
            pages.append(page)

    if load_errors:
        display(pd.DataFrame(load_errors))
        raise RuntimeError("At least one PDF failed to load; see load_errors above.")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    chunks = text_splitter.split_documents(pages)

    print(f"Loaded pages: {len(pages):,}")
    print(f"Created chunks: {len(chunks):,}")
    print("Example chunk metadata:")
    print(chunks[0].metadata)

    vectorstore = FAISS.from_documents(chunks, embedding_model)
    VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)
    vectorstore.save_local(str(VECTORSTORE_DIR))

    manifest = {
        "embedding_model": EMBEDDING_MODEL_NAME,
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "filtered_dataset_rows": int(len(df_task3)),
        "pdf_count": int(len(relevant_docs)),
        "page_count": int(len(pages)),
        "chunk_count": int(len(chunks)),
    }
    VECTORSTORE_MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))
    print(f"Saved FAISS index to {VECTORSTORE_DIR}")
    print(json.dumps(manifest, indent=2))


Filtered dataset rows: 100
Relevant PDFs: 42


,doc_name,company,doc_period,repo_pdf_filename
0,3M_2022_10K,3M,2022,3M_2022_10K.pdf
1,3M_2023Q2_10Q,3M,2023,3M_2023Q2_10Q.pdf
2,ADOBE_2022_10K,Adobe,2022,ADOBE_2022_10K.pdf
3,AES_2022_10K,AES Corporation,2022,AES_2022_10K.pdf
4,AMCOR_2022_8K_dated-2022-07-01,Amcor,2022,AMCOR_2022_8K_dated-2022-07-01.pdf
5,AMCOR_2023Q2_10Q,Amcor,2023,AMCOR_2023Q2_10Q.pdf
6,AMCOR_2023Q4_EARNINGS,Amcor,2023,AMCOR_2023Q4_EARNINGS.pdf
7,AMCOR_2023_10K,Amcor,2023,AMCOR_2023_10K.pdf
8,AMD_2022_10K,AMD,2022,AMD_2022_10K.pdf
9,AMERICANEXPRESS_2022_10K,American Express,2022,AMERICANEXPRESS_2022_10K.pdf


PDFs available in data/financebench_pdfs: 42


Loaded existing FAISS index from data/vectorstore/financebench_bge_small_v1_5
{
  "embedding_model": "BAAI/bge-small-en-v1.5",
  "chunk_size": 1000,
  "chunk_overlap": 150,
  "filtered_dataset_rows": 100,
  "pdf_count": 42,
  "page_count": 5448,
  "chunk_count": 24218
}


In [13]:
def parse_list_value(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    return value

def token_set(text: str) -> set[str]:
    tokens = re.findall(r"[a-z0-9$%.,/-]+", str(text).lower())
    return {token for token in tokens if len(token) > 2 and token not in STOPWORDS}

def evidence_recall(chunk_text: str, evidence_texts: list[str]) -> float:
    chunk_tokens = token_set(chunk_text)
    best = 0.0
    for evidence_text in evidence_texts:
        evidence_tokens = token_set(evidence_text)
        if not evidence_tokens:
            continue
        best = max(best, len(chunk_tokens & evidence_tokens) / len(evidence_tokens))
    return round(best, 3)

def preview_text(text: str, max_chars: int = 320) -> str:
    compact = " ".join(str(text).split())
    return compact[:max_chars] + ("..." if len(compact) > max_chars else "")

sample_questions = (
    df_task3[df_task3["financebench_id"].isin(TASK3_SAMPLE_IDS)]
    .set_index("financebench_id")
    .loc[TASK3_SAMPLE_IDS]
    .reset_index()
)

retrieval_rows = []
for _, question_row in sample_questions.iterrows():
    expected_doc = question_row["doc_name"]
    expected_pages = parse_list_value(question_row["evidence_page_nums"])
    evidence_texts = parse_list_value(question_row["evidence_texts"])
    retrieved_docs = vectorstore.similarity_search(question_row["question"], k=TASK3_TOP_K)

    for rank, chunk in enumerate(retrieved_docs, start=1):
        metadata = chunk.metadata
        retrieved_page = metadata.get("page_number")
        retrieval_rows.append(
            {
                "financebench_id": question_row["financebench_id"],
                "rank": rank,
                "expected_doc": expected_doc,
                "retrieved_doc": metadata.get("doc_name"),
                "doc_match": metadata.get("doc_name") == expected_doc,
                "expected_pages": expected_pages,
                "retrieved_page_number": retrieved_page,
                "page_match": retrieved_page in expected_pages,
                "evidence_token_recall": evidence_recall(chunk.page_content, evidence_texts),
                "question": question_row["question"],
                "chunk_preview": preview_text(chunk.page_content),
            }
        )

retrieval_check_df = pd.DataFrame(retrieval_rows)
display(
    retrieval_check_df[
        [
            "financebench_id",
            "rank",
            "doc_match",
            "page_match",
            "evidence_token_recall",
            "expected_doc",
            "retrieved_doc",
            "expected_pages",
            "retrieved_page_number",
            "chunk_preview",
        ]
    ]
)

retrieval_summary_df = (
    retrieval_check_df.groupby("financebench_id")
    .agg(
        question=("question", "first"),
        expected_doc=("expected_doc", "first"),
        retrieved_docs=("retrieved_doc", lambda values: list(dict.fromkeys(values))),
        expected_pages=("expected_pages", "first"),
        retrieved_pages=("retrieved_page_number", list),
        any_right_doc=("doc_match", "any"),
        any_right_page=("page_match", "any"),
        best_evidence_token_recall=("evidence_token_recall", "max"),
    )
    .reset_index()
)

display(retrieval_summary_df)


,financebench_id,rank,doc_match,page_match,evidence_token_recall,expected_doc,retrieved_doc,expected_pages,retrieved_page_number,chunk_preview
0,financebench_id_00005,1,True,False,0.158,CORNING_2022_10K,CORNING_2022_10K,[59],101,(1) Corning obtained a controlling interest in...
1,financebench_id_00005,2,True,False,0.063,CORNING_2022_10K,CORNING_2022_10K,[59],102,(1) Corning obtained a controlling interest in...
2,financebench_id_00005,3,False,False,0.137,CORNING_2022_10K,3M_2022_10K,[59],37,not defined under U.S. generally accepted acco...
3,financebench_id_00005,4,False,False,0.074,CORNING_2022_10K,3M_2023Q2_10Q,[59],70,"Current assets $ 15,754 $ 14,688 $ 1,066 Less:..."
4,financebench_id_00005,5,True,False,0.053,CORNING_2022_10K,CORNING_2022_10K,[59],90,Corning uses regression analysis or the critic...
5,financebench_id_00283,1,False,False,0.250,Pfizer_2023Q2_10Q,PFIZER_2021_10K,[40],70,"to the UpjohnBusiness, with the remainder cons..."
6,financebench_id_00283,2,False,False,0.179,Pfizer_2023Q2_10Q,PFIZER_2021_10K,[40],75,Transforming to a More Focused Company Program...
7,financebench_id_00283,3,False,False,0.071,Pfizer_2023Q2_10Q,PFIZER_2021_10K,[40],109,"1,731 $926 $810 CONSUMER HEALTHCARE BUSINESS$ ..."
8,financebench_id_00283,4,False,False,0.107,Pfizer_2023Q2_10Q,PFIZER_2021_10K,[40],70,"as anindependent publicly traded company, whic..."
9,financebench_id_00283,5,True,True,1.000,Pfizer_2023Q2_10Q,Pfizer_2023Q2_10Q,[40],40,Biopharma is the only reportable segment. See ...


,financebench_id,question,expected_doc,retrieved_docs,expected_pages,retrieved_pages,any_right_doc,any_right_page,best_evidence_token_recall
0,financebench_id_00005,Does Corning have positive working capital bas...,CORNING_2022_10K,"[CORNING_2022_10K, 3M_2022_10K, 3M_2023Q2_10Q]",[59],"[101, 102, 37, 70, 90]",True,False,0.158
1,financebench_id_00283,How much does Pfizer expect to pay to spin off...,Pfizer_2023Q2_10Q,"[PFIZER_2021_10K, Pfizer_2023Q2_10Q]",[40],"[70, 75, 109, 70, 40]",True,True,1.000
2,financebench_id_00288,Was there any drop in Cash & Cash equivalents ...,BESTBUY_2024Q2_10Q,"[BESTBUY_2023_10K, BESTBUY_2024Q2_10Q, JOHNSON...",[19],"[28, 19, 36, 88, 61]",True,True,0.091


### Task 3 Observations

This saved run confirmed that the 42 FinanceBench PDFs referenced by the filtered dataset are available locally and then loaded the existing FAISS index at `data/vectorstore/financebench_bge_small_v1_5`. The manifest for that index covers only the filtered corpus: 42 FinanceBench PDFs, 5,448 loaded pages, and 24,218 chunks using `chunk_size=1000` and `chunk_overlap=150`. The same saved index is reused by the later RAG, evaluation, and bonus cells.

The top-5 retrieval audit was mixed. For `financebench_id_00005` (Corning working capital), retrieval found the right document at ranks 1, 2, and 5, but it did **not** retrieve the evidence page 59; the best evidence-token recall was only 0.158. For `financebench_id_00283` (Pfizer / Upjohn), the exact evidence chunk from `Pfizer_2023Q2_10Q` page 40 appeared at rank 5 with evidence-token recall 1.000, but ranks 1-4 came from `PFIZER_2021_10K`, so a smaller `k` would miss the right filing. For `financebench_id_00288` (Best Buy cash), the right Q2 2024 evidence page appeared at rank 2 even though the question does not name Best Buy, but rank 1 came from the FY2023 10-K.

The main pattern is that query-only semantic retrieval often gets close, but it can confuse filings for the same company across periods or retrieve financially similar pages from other companies. For Tasks 4-7, the baseline intentionally does **not** use gold dataset metadata such as the expected `doc_name`, `company`, or `doc_period` as retrieval hints. Evaluating `page-hit@k` is therefore important because relevant evidence can appear below rank 1 or be missed entirely.


## Task 4 - RAG Pipeline

This section reloads the saved FAISS index from Task 3, retrieves relevant chunks for a user query, formats them as context, and sends the query plus context to `meta-llama/Llama-3.3-70B-Instruct` through the Nebius OpenAI-compatible API.


In [14]:
def get_nebius_client() -> OpenAI:
    if load_dotenv is not None:
        load_dotenv(dotenv_path=REPO_ROOT / ".env", override=False)

    api_key = os.getenv("NEBIUS_API_KEY")
    if not api_key:
        api_key = getpass("Nebius API key: ")
        if api_key:
            os.environ["NEBIUS_API_KEY"] = api_key

    if not api_key:
        raise ValueError("Set NEBIUS_API_KEY in .env or enter a Nebius API key to run RAG generation.")
    ensure_api_ca_bundle()
    return OpenAI(base_url=NEBIUS_BASE_URL, api_key=api_key, timeout=60.0, max_retries=API_MAX_RETRIES)

def load_rag_vectorstore() -> FAISS:
    if not (RAG_VECTORSTORE_DIR / "index.faiss").exists() or not (RAG_VECTORSTORE_DIR / "index.pkl").exists():
        raise FileNotFoundError(f"FAISS index not found at {RAG_VECTORSTORE_DIR}. Run Task 3 first.")

    embeddings = HuggingFaceEmbeddings(
        model_name=RAG_EMBEDDING_MODEL_NAME,
        model_kwargs={"device": LOCAL_TORCH_DEVICE},
        encode_kwargs={"normalize_embeddings": True},
    )
    return FAISS.load_local(
        str(RAG_VECTORSTORE_DIR),
        embeddings,
        allow_dangerous_deserialization=True,
    )

rag_vectorstore = load_rag_vectorstore()
print(f"Loaded FAISS vector store from {RAG_VECTORSTORE_DIR}")

def prompt_terms(text: str) -> set[str]:
    terms = set(re.findall(r"[A-Za-z0-9][A-Za-z0-9&._/-]*", str(text).lower()))
    return {term for term in terms if term not in STOPWORDS and len(term) > 1}

def build_rag_user_prompt(query: str, context: str) -> str:
    return f"""Question:
{query}

Retrieved FinanceBench context:
{context}

Instructions:
- Answer using only the retrieved context above.
- Prefer chunks whose company, filing name, fiscal period, and date match the question.
- Ignore mismatched-company or mismatched-period chunks unless the question asks for a comparison.
- For numeric questions, use Answer / Unit / Period/date / Formula / Evidence fields.
- If the answer is not present, say exactly: The context does not contain enough information.
- Cite each factual sentence with [doc_name, page N]."""

def build_rag_messages(system_prompt: str, user_prompt: str, few_shot_messages: "list[dict] | None" = None) -> list[dict]:
    messages = [{"role": "system", "content": system_prompt}]
    if few_shot_messages is None:
        few_shot_messages = RAG_FEW_SHOT_MESSAGES
    messages.extend(few_shot_messages)
    messages.append({"role": "user", "content": user_prompt})
    return messages

def format_chunk_for_prompt(chunk, rank: int) -> str:
    metadata = chunk.metadata
    doc_name = metadata.get("doc_name", "unknown_doc")
    page_number = metadata.get("page_number", "unknown_page")
    company = metadata.get("company", "unknown_company")
    doc_period = metadata.get("doc_period", "unknown_period")
    text = " ".join(chunk.page_content.split())

    return (
        f"--- Chunk {rank} ---\n"
        f"doc_name: {doc_name}\n"
        f"company: {company}\n"
        f"doc_period: {doc_period}\n"
        f"page_number: {page_number}\n"
        f"content:\n{text}"
    )

def format_retrieved_context(retrieved_docs: list) -> str:
    if not retrieved_docs:
        return "NO_RELEVANT_CONTEXT_RETRIEVED"
    return "\n\n".join(format_chunk_for_prompt(chunk, rank) for rank, chunk in enumerate(retrieved_docs, start=1))

def serialize_retrieved_chunk(chunk, rank: int) -> dict:
    metadata = chunk.metadata
    return {
        "rank": rank,
        "doc_name": metadata.get("doc_name"),
        "company": metadata.get("company"),
        "doc_period": metadata.get("doc_period"),
        "page_number": metadata.get("page_number"),
        "content": chunk.page_content,
    }

def answer_with_rag(query: str, k: int = 4) -> dict:
    """Retrieve FinanceBench chunks using only the user query and answer with the Nebius Llama model."""
    if not str(query).strip():
        raise ValueError("query must be a non-empty string")

    k = max(0, int(k))
    retrieved_docs = rag_vectorstore.similarity_search(query, k=k) if k > 0 else []
    context = format_retrieved_context(retrieved_docs)
    user_prompt = build_rag_user_prompt(query, context)
    client = get_nebius_client()

    def _request():
        response = client.chat.completions.create(
            model=RAG_GENERATION_MODEL,
            messages=build_rag_messages(RAG_SYSTEM_PROMPT, user_prompt),
            temperature=0,
            max_tokens=700,
        )
        return response.choices[0].message.content.strip()

    answer = call_with_retries(_request, "RAG generation")

    return {
        "answer": answer,
        "retrieved_chunks": [
            serialize_retrieved_chunk(chunk, rank)
            for rank, chunk in enumerate(retrieved_docs, start=1)
        ],
    }


Loaded FAISS vector store from data/vectorstore/financebench_bge_small_v1_5


### Task 4 Usage

Run `answer_with_rag("your question", k=4)` to generate an answer. The function asks for `NEBIUS_API_KEY` if it is not already set in the environment. The returned dictionary contains the model answer plus the retrieved chunks used as context, including each chunk's `doc_name` and `page_number` metadata. Retrieval uses only the user query; dataset labels such as the gold `doc_name`, `company`, and `doc_period` are not passed into the retriever or prompt.


## Task 5 - Run and Compare

Run the same 10 questions from Task 1 through the RAG pipeline, overwrite the raw generated-answer artifact, and export the side-by-side comparison to Excel.


In [15]:
if "answer_with_rag" not in globals():
    raise RuntimeError("Run the Task 4 RAG pipeline cell before running Task 5.")
if "naive_generation_df" not in globals():
    raise RuntimeError("Run the Task 1 generation cell before running Task 5.")

task5_questions_df = naive_generation_df.copy()

# Keep the exact Task 1 ordering: first five domain-relevant and first five novel-generated.
task5_questions_df = task5_questions_df[
    [
        "financebench_id",
        "question_type",
        "question",
        "naive_answer",
        "ground_truth",
        "doc_name",
        "evidence_page_nums",
    ]
].copy()


def load_task5_cached_rows(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = json.loads(path.read_text())
    if not isinstance(rows, list):
        raise ValueError(f"Expected a JSON list in {path}, found {type(rows).__name__}")
    return rows


def has_complete_task5_cache(rows: list[dict], expected_ids: list[str]) -> bool:
    if len(rows) != len(expected_ids):
        return False
    by_id = {row.get("financebench_id"): row for row in rows if isinstance(row, dict)}
    if set(by_id) != set(expected_ids):
        return False
    return all(
        isinstance(by_id[financebench_id].get("RAG_answer"), str)
        and isinstance(by_id[financebench_id].get("retrieved_chunks"), list)
        for financebench_id in expected_ids
    )


task5_expected_ids = task5_questions_df["financebench_id"].tolist()
task5_order = {financebench_id: index for index, financebench_id in enumerate(task5_expected_ids)}

if TASK5_RAW_PATH.exists() and not FORCE_TASK5_REGEN:
    cached_task5_rows = load_task5_cached_rows(TASK5_RAW_PATH)
    if has_complete_task5_cache(cached_task5_rows, task5_expected_ids):
        task5_rag_rows = sorted(cached_task5_rows, key=lambda row: task5_order[row["financebench_id"]])
        print(f"Loaded cached Task 5 RAG answers from {TASK5_RAW_PATH}")
    else:
        print(f"Cached Task 5 artifact at {TASK5_RAW_PATH} is incomplete; regenerating it.")
        load_dotenv(REPO_ROOT / ".env")
        if not os.getenv("NEBIUS_API_KEY"):
            raise ValueError("NEBIUS_API_KEY is not set. Add it to .env before running Task 5.")

        def task5_rag_row(item: tuple[int, pd.Series]) -> dict:
            _, row = item
            rag_result = answer_with_rag(row["question"], k=TASK5_RAG_K)
            retrieved_chunks = rag_result["retrieved_chunks"]
            return {
                "financebench_id": row["financebench_id"],
                "RAG_answer": rag_result["answer"],
                "retrieved_chunks": retrieved_chunks,
                "retrieved_doc_names": [chunk.get("doc_name") for chunk in retrieved_chunks],
                "retrieved_page_numbers": [chunk.get("page_number") for chunk in retrieved_chunks],
            }

        task5_rag_rows = run_parallel_jobs(
            task5_questions_df.iterrows(),
            task5_rag_row,
            desc="Task 5 RAG",
            max_workers=API_GENERATION_WORKERS,
            checkpoint_path=TASK5_RAW_PATH,
        )
        TASK5_RAW_PATH.write_text(json.dumps(task5_rag_rows, indent=2, ensure_ascii=False))
        print(f"Saved raw RAG answers to {TASK5_RAW_PATH}")
else:
    load_dotenv(REPO_ROOT / ".env")
    if not os.getenv("NEBIUS_API_KEY"):
        raise ValueError("NEBIUS_API_KEY is not set. Add it to .env before running Task 5.")

    def task5_rag_row(item: tuple[int, pd.Series]) -> dict:
        _, row = item
        rag_result = answer_with_rag(row["question"], k=TASK5_RAG_K)
        retrieved_chunks = rag_result["retrieved_chunks"]
        return {
            "financebench_id": row["financebench_id"],
            "RAG_answer": rag_result["answer"],
            "retrieved_chunks": retrieved_chunks,
            "retrieved_doc_names": [chunk.get("doc_name") for chunk in retrieved_chunks],
            "retrieved_page_numbers": [chunk.get("page_number") for chunk in retrieved_chunks],
        }

    task5_rag_rows = run_parallel_jobs(
        task5_questions_df.iterrows(),
        task5_rag_row,
        desc="Task 5 RAG",
        max_workers=API_GENERATION_WORKERS,
        checkpoint_path=TASK5_RAW_PATH,
    )
    TASK5_RAW_PATH.write_text(json.dumps(task5_rag_rows, indent=2, ensure_ascii=False))
    print(f"Saved raw RAG answers to {TASK5_RAW_PATH}")

task5_rag_df = pd.DataFrame(task5_rag_rows)
assignment_2_run_and_compare = task5_questions_df.merge(task5_rag_df, on="financebench_id", how="left")
assignment_2_run_and_compare[TASK5_REQUIRED_COLUMNS].to_excel(TASK5_XLSX_PATH, index=False)
print(f"Saved Task 5 deliverable to {TASK5_XLSX_PATH}")

display(assignment_2_run_and_compare[TASK5_REQUIRED_COLUMNS])
display(
    assignment_2_run_and_compare[
        [
            "financebench_id",
            "doc_name",
            "evidence_page_nums",
            "retrieved_doc_names",
            "retrieved_page_numbers",
        ]
    ]
)


Loaded cached Task 5 RAG answers from rag_faiss/artifacts/task5_rag_answers_raw.json


Saved Task 5 deliverable to rag_faiss/outputs/assignment_2_run_and_compare.xlsx


,financebench_id,question_type,question,naive_answer,RAG_answer,ground_truth
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,The answer cannot be determined from the quest...,The context does not contain enough information.,Yes. Corning had a positive working capital am...
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,The answer cannot be determined from the quest...,The context does not contain enough information.,"No, American Water Works had negative working ..."
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,The answer cannot be determined from the quest...,The context does not contain enough information.,Yes. Paypal has a positive working capital of ...
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,Gross margins are not a relevant metric for a ...,The context does not contain enough information.,"Since JPM is a financial institution, gross ma..."
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,The answer cannot be determined from the quest...,"Answer: Yes, Verizon is a capital intensive bu...",Yes. Verizon's capital intensity ratio was app...
5,financebench_id_00283,novel-generated,How much does Pfizer expect to pay to spin off...,The answer cannot be determined from the quest...,Answer: $700 million. Unit: USD millions. Peri...,77.78
6,financebench_id_00288,novel-generated,Was there any drop in Cash & Cash equivalents ...,The question does not provide the Cash & Cash ...,The context does not contain enough information.,"Yes, there was a decline of ~42% between FY202..."
7,financebench_id_00299,novel-generated,Which of JPM's business segments had the lowes...,The answer cannot be determined from the quest...,The context does not contain enough information.,Corporate. Its net revenue was -$473 million.
8,financebench_id_00302,novel-generated,Did Pfizer grow its PPNE between FY20 and FY21?,The question does not provide the PPNE (Pfizer...,The context does not contain enough information.,"Yes, change in PPNE was positive year over year"
9,financebench_id_00382,novel-generated,Which region had the Highest EBITDAR Contribut...,The answer cannot be determined from the quest...,Answer: Las Vegas Strip Resorts had the highes...,Las Vegas resorts contributed ~90% of company ...


,financebench_id,doc_name,evidence_page_nums,retrieved_doc_names,retrieved_page_numbers
0,financebench_id_00005,CORNING_2022_10K,[59],"[CORNING_2022_10K, CORNING_2022_10K, 3M_2022_1...","[101, 102, 37, 70, 90]"
1,financebench_id_00070,AMERICANWATERWORKS_2022_10K,"[80, 81]","[AMERICANWATERWORKS_2022_10K, AMERICANWATERWOR...","[81, 139, 144, 37, 138]"
2,financebench_id_00080,PAYPAL_2022_10K,[60],"[PAYPAL_2022_10K, PAYPAL_2022_10K, PAYPAL_2022...","[7, 3, 32, 48, 75]"
3,financebench_id_00206,JPMORGAN_2022_10K,[2],"[BESTBUY_2023_10K, JPMORGAN_2022Q2_10Q, AMD_20...","[17, 37, 45, 50, 29]"
4,financebench_id_00215,VERIZON_2022_10K,"[22, 55]","[VERIZON_2021_10K, VERIZON_2021_10K, VERIZON_2...","[35, 19, 22, 16, 115]"
5,financebench_id_00283,Pfizer_2023Q2_10Q,[40],"[PFIZER_2021_10K, PFIZER_2021_10K, PFIZER_2021...","[70, 75, 109, 70, 40]"
6,financebench_id_00288,BESTBUY_2024Q2_10Q,[19],"[BESTBUY_2023_10K, BESTBUY_2024Q2_10Q, JOHNSON...","[28, 19, 36, 88, 61]"
7,financebench_id_00299,JPMORGAN_2021Q1_10Q,[18],"[JPMORGAN_2021Q1_10Q, BESTBUY_2023_10K, JPMORG...","[4, 61, 176, 122, 22]"
8,financebench_id_00302,PFIZER_2021_10K,[58],"[PFIZER_2021_10K, PFIZER_2021_10K, PFIZER_2021...","[135, 133, 134, 110, 40]"
9,financebench_id_00382,MGMRESORTS_2022Q4_EARNINGS,[12],"[MGMRESORTS_2022_10K, MGMRESORTS_2022_10K, MGM...","[40, 40, 31, 99, 3]"


### Task 5 Discussion

**Did RAG help?** In the saved Task 5 outputs, RAG clearly helped on `financebench_id_00215`: the naive model refused, while RAG retrieved Verizon context and answered that Verizon is capital intensive. `financebench_id_00382` also improved from a refusal to an on-topic region-level answer about Las Vegas Strip Resorts, although the wording did not fully match the gold answer and the citations pointed to `MGMRESORTS_2022_10K` rather than the gold earnings document.

**Did RAG hurt or still fail?** Most of the saved RAG answers still failed. For `financebench_id_00005`, `financebench_id_00070`, `financebench_id_00080`, `financebench_id_00288`, `financebench_id_00299`, and `financebench_id_00302`, the model still answered `The context does not contain enough information.` even though some of those questions do have supporting evidence in the corpus. For `financebench_id_00206`, naive generation correctly identified that gross margin is not a relevant bank metric, while RAG regressed to a refusal. For `financebench_id_00283`, RAG retrieved a Pfizer page but answered `$700 million` instead of the gold `$77.78 million`, so retrieval/generation did not recover the exact required amount.

**Patterns by question type.** In these saved results, gains from RAG were limited rather than broad. The `domain-relevant` set was still weak because the working-capital questions require exact balance-sheet rows and arithmetic, and one conceptual question (`financebench_id_00206`) actually regressed from correct naive reasoning to a RAG refusal. The `novel-generated` set showed mixed behavior: MGM improved to an on-topic answer, Pfizer/Upjohn produced a wrong number, and Best Buy, JPM segment revenue, and Pfizer PPNE still ended as refusals.


## Task 6 - Evaluation

Evaluate the RAG pipeline at three levels: final-answer correctness with an LLM judge, faithfulness with Ragas, and retrieval page-hit rate. The FinanceBench paper emphasizes that shared vector stores are hard because retrieval quality and downstream reasoning both matter, so this section reports component-level numbers rather than only answer correctness.


In [16]:
load_dotenv(REPO_ROOT / ".env")

if "answer_with_rag" not in globals() or "rag_vectorstore" not in globals():
    raise RuntimeError("Run the Task 4 RAG pipeline cell before running Task 6.")

print(f"Configured Nebius judge model for this notebook run: {TASK6_JUDGE_MODEL}")

task6_df = pd.read_json(TASK6_DATASET_PATH, lines=True)
task6_df = task6_df.sort_values("financebench_id", kind="stable").reset_index(drop=True)
task6_expected_ids = task6_df["financebench_id"].tolist()
faithfulness_expected_ids = task6_df.head(TASK6_FAITHFULNESS_LIMIT)["financebench_id"].tolist()
print(f"Task 6 examples: {len(task6_df):,}")


def save_json_rows(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(exist_ok=True)
    path.write_text(json.dumps(rows, indent=2, ensure_ascii=False))


def load_json_rows(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = json.loads(path.read_text())
    if not isinstance(rows, list):
        raise ValueError(f"Expected a JSON list in {path}, found {type(rows).__name__}")
    return rows


def order_rows_by_expected_ids(rows: list[dict], expected_ids: list[str]) -> list[dict]:
    by_id = {row["financebench_id"]: row for row in rows}
    return [by_id[financebench_id] for financebench_id in expected_ids]


def as_list(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    return value


def get_expected_pages(row: pd.Series) -> set[int]:
    return {int(page) for page in as_list(row.get("evidence_page_nums")) if str(page).strip() != ""}


def page_hit_from_chunks(chunks: list[dict], expected_doc: str, expected_pages: set[int], k: int) -> int:
    if not expected_pages:
        return 0
    for chunk in chunks[:k]:
        if chunk.get("doc_name") == expected_doc and chunk.get("page_number") in expected_pages:
            return 1
    return 0


def _truncate_at_word_boundary(text: str, max_chars: int) -> str:
    text = " ".join(str(text).split())
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rsplit(" ", 1)[0]


def extract_relevant_evidence_span(content: str, query: "str | None" = None, max_chars: int = TASK6_FAITHFULNESS_CONTEXT_MAX_CHARS) -> str:
    content = " ".join(str(content).split())
    if not query or len(content) <= max_chars:
        return _truncate_at_word_boundary(content, max_chars)

    query_tokens = prompt_terms(query)
    sentences = [sentence.strip() for sentence in re.split(r"(?<=[.!?])\s+|\n+", content) if sentence.strip()]
    if not sentences:
        return _truncate_at_word_boundary(content, max_chars)

    scored = []
    for index, sentence in enumerate(sentences):
        sentence_tokens = prompt_terms(sentence)
        overlap = len(query_tokens & sentence_tokens)
        numeric_bonus = 1 if re.search(r"\d", sentence) and re.search(r"\d|how much|percent|ratio|increase|decrease", str(query).lower()) else 0
        scored.append((overlap + numeric_bonus, index, sentence))
    scored.sort(key=lambda item: (item[0], -item[1]), reverse=True)

    selected_indices = sorted(index for score, index, _ in scored[:4] if score > 0)
    if not selected_indices:
        selected_indices = [0]

    selected = []
    for index in selected_indices:
        selected.append(sentences[index])
        candidate = " ".join(selected)
        if len(candidate) >= max_chars:
            break
    return _truncate_at_word_boundary(" ".join(selected), max_chars)


def build_contexts_from_chunks(chunks: list[dict], query: "str | None" = None, max_chars: int = TASK6_FAITHFULNESS_CONTEXT_MAX_CHARS) -> list[str]:
    contexts = []
    for chunk in chunks:
        doc_name = chunk.get("doc_name")
        page_number = chunk.get("page_number")
        content = extract_relevant_evidence_span(chunk.get("content", ""), query=query, max_chars=max_chars)
        contexts.append(f"doc_name: {doc_name}\npage_number: {page_number}\ncontent:\n{content}")
    return contexts


def has_complete_rag_rows(rows: list[dict], expected_ids: list[str]) -> bool:
    if len(rows) != len(expected_ids):
        return False
    by_id = {row.get("financebench_id"): row for row in rows if isinstance(row, dict)}
    if set(by_id) != set(expected_ids):
        return False
    return all(
        isinstance(by_id[financebench_id].get("rag_answer"), str)
        and isinstance(by_id[financebench_id].get("retrieved_chunks"), list)
        for financebench_id in expected_ids
    )


# 1) Run all filtered-dataset questions through the RAG pipeline and overwrite the raw output artifact.
def task6_rag_row(item: tuple[int, pd.Series]) -> dict:
    _, row = item
    financebench_id = row["financebench_id"]
    rag_result = answer_with_rag(row["question"], k=TASK6_RAG_K)
    return {
        "financebench_id": financebench_id,
        "question": row["question"],
        "answer": row["answer"],
        "rag_answer": rag_result["answer"],
        "retrieved_chunks": rag_result["retrieved_chunks"],
        "retrieved_doc_names": [chunk.get("doc_name") for chunk in rag_result["retrieved_chunks"]],
        "retrieved_page_numbers": [chunk.get("page_number") for chunk in rag_result["retrieved_chunks"]],
    }


# 2) Correctness evaluation with a DeepSeek judge.
correctness_system_prompt = """You are a FinanceBench binary answer judge.
Return exactly one minified JSON object and nothing else.
The response must be valid JSON and must match this schema:
{"verdict":"correct|incorrect","justification":"one short sentence"}
Example JSON output:
{"verdict":"correct","justification":"The model answer is semantically equivalent to the gold answer."}
Use "correct" only when the model answer is semantically equivalent to the gold answer, allowing minor rounding or equivalent wording.
Use "incorrect" for refusals, missing answers, wrong numbers, wrong direction, unsupported claims, or contradictions.
Do not include markdown, code fences, or extra keys."""


def parse_judge_json(text: str) -> dict:
    text = str(text).strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{[^{}]*\}", text, flags=re.DOTALL)
        if not match:
            raise ValueError(f"Judge did not return JSON: {text[:240]}")
        parsed = json.loads(match.group(0))
    if not isinstance(parsed, dict):
        raise ValueError(f"Judge JSON was not an object: {parsed!r}")
    verdict = str(parsed.get("verdict", "")).strip().lower()
    if verdict not in {"correct", "incorrect"}:
        raise ValueError(f"Judge returned invalid verdict: {verdict!r}")
    justification = str(parsed.get("justification", "")).strip()
    if not justification:
        raise ValueError("Judge returned an empty justification")
    return {"verdict": verdict, "justification": justification}


def judge_correctness(client: OpenAI, question: str, gold_answer: str, model_answer: str) -> dict:
    user_prompt = f"""Evaluate whether the model answer matches the gold answer.

Question:
{question}

Gold answer:
{gold_answer}

Model answer:
{model_answer}

Return exactly one valid JSON object now."""

    def _request():
        response = client.chat.completions.create(
            model=TASK6_JUDGE_MODEL,
            messages=[
                {"role": "system", "content": correctness_system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0,
            max_tokens=220,
            response_format={"type": "json_object"},
        )
        content = response.choices[0].message.content.strip()
        return parse_judge_json(content)

    return call_with_retries(_request, "Correctness judge")


def has_valid_correctness_record(record: dict) -> bool:
    if not record:
        return False
    if record.get("judge_model") != TASK6_JUDGE_MODEL:
        return False
    if record.get("judge_run_version") != TASK6_JUDGE_RUN_VERSION:
        return False
    if record.get("correctness") not in {"correct", "incorrect"}:
        return False
    return bool(str(record.get("correctness_justification", "")).strip())


support_system_prompt = """You are a FinanceBench answer support verifier.
Return exactly one minified JSON object and nothing else.
The response must be valid JSON and must match this schema:
{"support_verdict":"supported|unsupported|insufficient_context","citation_status":"valid|invalid|missing|not_applicable","numeric_status":"valid|invalid|not_applicable","justification":"one short sentence"}
Example JSON output:
{"support_verdict":"supported","citation_status":"valid","numeric_status":"valid","justification":"The answer's numeric claim and citation are supported by the retrieved context."}
Use "supported" only when every factual and numeric claim in the model answer is directly supported by the retrieved context.
Use "unsupported" when the answer includes facts, numbers, periods, directions, or citations not supported by the retrieved context.
Use "insufficient_context" when the retrieved context itself does not contain enough evidence to answer the question.
Do not include markdown, code fences, or extra keys."""


def parse_support_json(text: str) -> dict:
    text = str(text).strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{[^{}]*\}", text, flags=re.DOTALL)
        if not match:
            raise ValueError(f"Support judge did not return JSON: {text[:240]}")
        parsed = json.loads(match.group(0))
    if not isinstance(parsed, dict):
        raise ValueError(f"Support judge JSON was not an object: {parsed!r}")

    support_verdict = str(parsed.get("support_verdict", "")).strip().lower()
    citation_status = str(parsed.get("citation_status", "")).strip().lower()
    numeric_status = str(parsed.get("numeric_status", "")).strip().lower()
    if support_verdict not in {"supported", "unsupported", "insufficient_context"}:
        raise ValueError(f"Support judge returned invalid support_verdict: {support_verdict!r}")
    if citation_status not in {"valid", "invalid", "missing", "not_applicable"}:
        raise ValueError(f"Support judge returned invalid citation_status: {citation_status!r}")
    if numeric_status not in {"valid", "invalid", "not_applicable"}:
        raise ValueError(f"Support judge returned invalid numeric_status: {numeric_status!r}")
    justification = str(parsed.get("justification", "")).strip()
    if not justification:
        raise ValueError("Support judge returned an empty justification")
    return {
        "support_verdict": support_verdict,
        "citation_status": citation_status,
        "numeric_status": numeric_status,
        "justification": justification,
    }


def judge_answer_support(client: OpenAI, question: str, model_answer: str, retrieved_chunks: list[dict]) -> dict:
    retrieved_contexts = build_contexts_from_chunks(
        retrieved_chunks,
        query=question,
        max_chars=TASK6_SUPPORT_CONTEXT_MAX_CHARS,
    )
    context_block = "\n\n---\n\n".join(retrieved_contexts) if retrieved_contexts else "NO_RELEVANT_CONTEXT_RETRIEVED"
    user_prompt = f"""Question:
{question}

Model answer:
{model_answer}

Retrieved context:
{context_block}

Verify whether the model answer is fully supported by the retrieved context. Check numeric values, units, periods/dates, directionality, and citations.
Return exactly one valid JSON object now."""

    def _request():
        response = client.chat.completions.create(
            model=TASK6_JUDGE_MODEL,
            messages=[
                {"role": "system", "content": support_system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0,
            max_tokens=360,
            response_format={"type": "json_object"},
        )
        content = response.choices[0].message.content.strip()
        return parse_support_json(content)

    return call_with_retries(_request, "Support judge")


def has_valid_support_record(record: dict) -> bool:
    if not record:
        return False
    if record.get("judge_model") != TASK6_JUDGE_MODEL:
        return False
    if record.get("support_run_version") != TASK6_SUPPORT_RUN_VERSION:
        return False
    if record.get("support_verdict") not in {"supported", "unsupported", "insufficient_context"}:
        return False
    if record.get("citation_status") not in {"valid", "invalid", "missing", "not_applicable"}:
        return False
    if record.get("numeric_status") not in {"valid", "invalid", "not_applicable"}:
        return False
    return bool(str(record.get("support_justification", "")).strip())


# 4) Faithfulness with Ragas collections API, synchronous .score(), first 20 examples only.
_task6_judge_client = None
_task6_faithfulness_metric = None


def require_nebius_api_key() -> str:
    api_key = os.getenv("NEBIUS_API_KEY")
    if not api_key:
        raise ValueError("NEBIUS_API_KEY is not set. Add it to .env before regenerating Task 6 artifacts.")
    return api_key


def get_task6_judge_client() -> OpenAI:
    global _task6_judge_client
    if _task6_judge_client is None:
        ensure_api_ca_bundle()
        _task6_judge_client = OpenAI(
            base_url=NEBIUS_BASE_URL,
            api_key=require_nebius_api_key(),
            timeout=60.0,
            max_retries=API_MAX_RETRIES,
        )
    return _task6_judge_client


def get_task6_faithfulness_metric() -> Faithfulness:
    global _task6_faithfulness_metric
    if _task6_faithfulness_metric is None:
        ensure_api_ca_bundle()
        ragas_client = AsyncOpenAI(
            base_url=NEBIUS_BASE_URL,
            api_key=require_nebius_api_key(),
            timeout=60.0,
            max_retries=API_MAX_RETRIES,
        )
        ragas_llm = llm_factory(TASK6_JUDGE_MODEL, client=ragas_client, temperature=0, max_tokens=TASK6_RAGAS_MAX_TOKENS)
        _task6_faithfulness_metric = Faithfulness(llm=ragas_llm)
    return _task6_faithfulness_metric


def score_faithfulness_with_sync_api(user_input: str, response: str, retrieved_contexts: list[str]) -> float:
    # Ragas .score() is synchronous and internally uses asyncio.run(). IPython kernels
    # often already have an event loop, so call .score() in a daemon worker thread
    # while still using the required synchronous API instead of .ascore().
    def _score_once():
        result_queue = queue.Queue(maxsize=1)
        faithfulness_metric = get_task6_faithfulness_metric()

        def _score():
            try:
                score = faithfulness_metric.score(
                    user_input=user_input,
                    response=response,
                    retrieved_contexts=retrieved_contexts,
                ).value
                result_queue.put(("ok", float(score)))
            except BaseException as exc:
                result_queue.put(("error", exc))

        worker = Thread(target=_score, daemon=True)
        worker.start()
        worker.join(TASK6_RAGAS_SCORE_TIMEOUT_SECONDS)
        if worker.is_alive():
            raise TimeoutError(f"Ragas faithfulness timed out after {TASK6_RAGAS_SCORE_TIMEOUT_SECONDS} seconds")
        status, payload = result_queue.get_nowait()
        if status == "error":
            raise payload
        return payload

    return call_with_retries(_score_once, "Ragas faithfulness", max_retries=TASK6_RAGAS_MAX_RETRIES)


def has_valid_faithfulness_record(record: dict) -> bool:
    if not record or record.get("faithfulness_error"):
        return False
    value = record.get("faithfulness")
    try:
        return not math.isnan(float(value))
    except Exception:
        return False


def records_match_expected_ids(rows: list[dict], expected_ids: list[str], validator) -> bool:
    if len(rows) != len(expected_ids):
        return False
    by_id = {row.get("financebench_id"): row for row in rows if isinstance(row, dict)}
    if set(by_id) != set(expected_ids):
        return False
    return all(validator(by_id[financebench_id]) for financebench_id in expected_ids)


cached_rag_rows = load_json_rows(TASK6_RAG_ALL_RAW_PATH) if TASK6_RAG_ALL_RAW_PATH.exists() and not FORCE_TASK6_RAG_REGEN else []
if has_complete_rag_rows(cached_rag_rows, task6_expected_ids):
    rag_rows = order_rows_by_expected_ids(cached_rag_rows, task6_expected_ids)
    print(f"Loaded cached Task 6 RAG answers from {TASK6_RAG_ALL_RAW_PATH}")
else:
    if cached_rag_rows:
        print(f"Cached Task 6 RAG artifact at {TASK6_RAG_ALL_RAW_PATH} is incomplete; regenerating it.")
    require_nebius_api_key()
    rag_rows = run_parallel_jobs(
        task6_df.iterrows(),
        task6_rag_row,
        desc="Task 6 RAG answers",
        max_workers=API_GENERATION_WORKERS,
        checkpoint_path=TASK6_RAG_ALL_RAW_PATH,
    )
    save_json_rows(TASK6_RAG_ALL_RAW_PATH, rag_rows)
    print(f"Saved Task 6 RAG answers to {TASK6_RAG_ALL_RAW_PATH}")

rag_by_id = {row["financebench_id"]: row for row in rag_rows}


def task6_correctness_row(item: tuple[int, pd.Series]) -> dict:
    _, row = item
    financebench_id = row["financebench_id"]
    rag_answer = rag_by_id[financebench_id]["rag_answer"]
    verdict = judge_correctness(get_task6_judge_client(), row["question"], row["answer"], rag_answer)
    return {
        "financebench_id": financebench_id,
        "judge_model": TASK6_JUDGE_MODEL,
        "judge_run_version": TASK6_JUDGE_RUN_VERSION,
        "correctness": verdict["verdict"],
        "correctness_justification": verdict["justification"],
    }


cached_correctness_rows = load_json_rows(TASK6_CORRECTNESS_RAW_PATH) if TASK6_CORRECTNESS_RAW_PATH.exists() and not FORCE_TASK6_CORRECTNESS_REGEN else []
if records_match_expected_ids(cached_correctness_rows, task6_expected_ids, has_valid_correctness_record):
    correctness_rows = order_rows_by_expected_ids(cached_correctness_rows, task6_expected_ids)
    print(f"Loaded cached Task 6 correctness judgments from {TASK6_CORRECTNESS_RAW_PATH}")
else:
    if cached_correctness_rows:
        print(f"Cached Task 6 correctness artifact at {TASK6_CORRECTNESS_RAW_PATH} is incomplete or uses a different judge model; regenerating it.")
    require_nebius_api_key()
    correctness_rows = run_parallel_jobs(
        task6_df.iterrows(),
        task6_correctness_row,
        desc="Correctness judge",
        max_workers=API_JUDGE_WORKERS,
        checkpoint_path=TASK6_CORRECTNESS_RAW_PATH,
    )
    correctness_rows = order_rows_by_expected_ids(correctness_rows, task6_expected_ids)
    save_json_rows(TASK6_CORRECTNESS_RAW_PATH, correctness_rows)
    print(f"Saved Task 6 correctness judgments to {TASK6_CORRECTNESS_RAW_PATH}")

correctness_by_id = {row["financebench_id"]: row for row in correctness_rows}


def task6_support_row(item: tuple[int, pd.Series]) -> dict:
    _, row = item
    financebench_id = row["financebench_id"]
    rag_record = rag_by_id[financebench_id]
    support = judge_answer_support(
        get_task6_judge_client(),
        row["question"],
        rag_record["rag_answer"],
        rag_record["retrieved_chunks"],
    )
    return {
        "financebench_id": financebench_id,
        "judge_model": TASK6_JUDGE_MODEL,
        "support_run_version": TASK6_SUPPORT_RUN_VERSION,
        "support_verdict": support["support_verdict"],
        "citation_status": support["citation_status"],
        "numeric_status": support["numeric_status"],
        "support_justification": support["justification"],
    }


cached_support_rows = load_json_rows(TASK6_SUPPORT_RAW_PATH) if TASK6_SUPPORT_RAW_PATH.exists() and not FORCE_TASK6_SUPPORT_REGEN else []
if records_match_expected_ids(cached_support_rows, task6_expected_ids, has_valid_support_record):
    support_rows = order_rows_by_expected_ids(cached_support_rows, task6_expected_ids)
    print(f"Loaded cached Task 6 support judgments from {TASK6_SUPPORT_RAW_PATH}")
else:
    if cached_support_rows:
        print(f"Cached Task 6 support artifact at {TASK6_SUPPORT_RAW_PATH} is incomplete or uses a different judge model; regenerating it.")
    require_nebius_api_key()
    support_rows = run_parallel_jobs(
        task6_df.iterrows(),
        task6_support_row,
        desc="Support judge",
        max_workers=API_JUDGE_WORKERS,
        checkpoint_path=TASK6_SUPPORT_RAW_PATH,
    )
    support_rows = order_rows_by_expected_ids(support_rows, task6_expected_ids)
    save_json_rows(TASK6_SUPPORT_RAW_PATH, support_rows)
    print(f"Saved Task 6 support judgments to {TASK6_SUPPORT_RAW_PATH}")

support_by_id = {row["financebench_id"]: row for row in support_rows}


def task6_faithfulness_row(item: tuple[int, pd.Series]) -> dict:
    _, row = item
    financebench_id = row["financebench_id"]
    rag_record = rag_by_id[financebench_id]
    retrieved_contexts = build_contexts_from_chunks(rag_record["retrieved_chunks"], query=row["question"])
    try:
        score = score_faithfulness_with_sync_api(
            user_input=row["question"],
            response=rag_record["rag_answer"],
            retrieved_contexts=retrieved_contexts,
        )
        error = None
    except Exception as exc:
        score = float("nan")
        error = repr(exc)
    return {
        "financebench_id": financebench_id,
        "judge_model": TASK6_JUDGE_MODEL,
        "judge_run_version": TASK6_JUDGE_RUN_VERSION,
        "faithfulness": score,
        "faithfulness_error": error,
    }


cached_faithfulness_rows = load_json_rows(TASK6_FAITHFULNESS_RAW_PATH) if TASK6_FAITHFULNESS_RAW_PATH.exists() and not FORCE_TASK6_FAITHFULNESS_REGEN else []
if records_match_expected_ids(cached_faithfulness_rows, faithfulness_expected_ids, has_valid_faithfulness_record):
    faithfulness_rows = order_rows_by_expected_ids(cached_faithfulness_rows, faithfulness_expected_ids)
    print(f"Loaded cached Task 6 faithfulness scores from {TASK6_FAITHFULNESS_RAW_PATH}")
else:
    if cached_faithfulness_rows:
        print(f"Cached Task 6 faithfulness artifact at {TASK6_FAITHFULNESS_RAW_PATH} is incomplete or invalid; regenerating it.")
    require_nebius_api_key()
    faithfulness_eval_df = task6_df.head(TASK6_FAITHFULNESS_LIMIT)
    faithfulness_rows = run_parallel_jobs(
        faithfulness_eval_df.iterrows(),
        task6_faithfulness_row,
        desc="Ragas faithfulness",
        max_workers=API_FAITHFULNESS_WORKERS,
        checkpoint_path=TASK6_FAITHFULNESS_RAW_PATH,
    )
    faithfulness_rows = order_rows_by_expected_ids(faithfulness_rows, faithfulness_expected_ids)
    save_json_rows(TASK6_FAITHFULNESS_RAW_PATH, faithfulness_rows)
    print(f"Saved Task 6 faithfulness scores to {TASK6_FAITHFULNESS_RAW_PATH}")

faithfulness_by_id = {row["financebench_id"]: row for row in faithfulness_rows}

# 5) Retrieval page-hit@k for k in {1, 3, 5} across the filtered dataset.
evaluation_rows = []
for _, row in task6_df.iterrows():
    financebench_id = row["financebench_id"]
    rag_record = rag_by_id[financebench_id]
    expected_pages = get_expected_pages(row)
    retrieved_chunks = rag_record["retrieved_chunks"]
    correctness_record = correctness_by_id[financebench_id]
    faithfulness_record = faithfulness_by_id.get(financebench_id, {})

    eval_row = {
        "financebench_id": financebench_id,
        "question": row["question"],
        "correctness": correctness_record["correctness"],
        "correctness_justification": correctness_record["correctness_justification"],
        "support_verdict": support_by_id[financebench_id]["support_verdict"],
        "support_citation_status": support_by_id[financebench_id]["citation_status"],
        "support_numeric_status": support_by_id[financebench_id]["numeric_status"],
        "support_justification": support_by_id[financebench_id]["support_justification"],
        "faithfulness": faithfulness_record.get("faithfulness", float("nan")),
    }
    for k in TASK6_PAGE_HIT_K_VALUES:
        eval_row[f"page_hit_at_{k}"] = page_hit_from_chunks(
            retrieved_chunks,
            expected_doc=row["doc_name"],
            expected_pages=expected_pages,
            k=k,
        )
    evaluation_rows.append(eval_row)

assignment_2_evaluation = pd.DataFrame(evaluation_rows)
assert assignment_2_evaluation["correctness"].isin({"correct", "incorrect"}).all()
assert assignment_2_evaluation["support_verdict"].isin({"supported", "unsupported", "insufficient_context"}).all()
assert int(assignment_2_evaluation["faithfulness"].notna().sum()) == TASK6_FAITHFULNESS_LIMIT
assignment_2_evaluation.to_excel(TASK6_XLSX_PATH, index=False)
print(f"Saved Task 6 deliverable to {TASK6_XLSX_PATH}")

aggregate_rows = []
aggregate_rows.append(
    {
        "metric": "average_correctness",
        "value": (assignment_2_evaluation["correctness"] == "correct").mean(),
        "n": int(assignment_2_evaluation["correctness"].notna().sum()),
    }
)
aggregate_rows.append(
    {
        "metric": "average_faithfulness_first_20",
        "value": assignment_2_evaluation.loc[assignment_2_evaluation["faithfulness"].notna(), "faithfulness"].mean(),
        "n": int(assignment_2_evaluation["faithfulness"].notna().sum()),
    }
)
aggregate_rows.append(
    {
        "metric": "support_supported_rate",
        "value": (assignment_2_evaluation["support_verdict"] == "supported").mean(),
        "n": int(assignment_2_evaluation["support_verdict"].notna().sum()),
    }
)
aggregate_rows.append(
    {
        "metric": "support_valid_citation_rate",
        "value": (assignment_2_evaluation["support_citation_status"] == "valid").mean(),
        "n": int(assignment_2_evaluation["support_citation_status"].notna().sum()),
    }
)
for k in TASK6_PAGE_HIT_K_VALUES:
    aggregate_rows.append(
        {
            "metric": f"page_hit_at_{k}",
            "value": assignment_2_evaluation[f"page_hit_at_{k}"].mean(),
            "n": int(assignment_2_evaluation[f"page_hit_at_{k}"].notna().sum()),
        }
    )

task6_aggregate_df = pd.DataFrame(aggregate_rows)
display(assignment_2_evaluation)
display(task6_aggregate_df)


Configured Nebius judge model for this notebook run: deepseek-ai/DeepSeek-V3.2
Task 6 examples: 100
Loaded cached Task 6 RAG answers from rag_faiss/artifacts/task6_rag_all_answers_raw.json
Loaded cached Task 6 correctness judgments from rag_faiss/artifacts/task6_correctness_raw.json
Loaded cached Task 6 support judgments from rag_faiss/artifacts/task6_support_raw.json
Loaded cached Task 6 faithfulness scores from rag_faiss/artifacts/task6_faithfulness_raw.json
Saved Task 6 deliverable to rag_faiss/outputs/assignment_2_evaluation.xlsx


,financebench_id,question,correctness,correctness_justification,support_verdict,support_citation_status,support_numeric_status,support_justification,faithfulness,page_hit_at_1,page_hit_at_3,page_hit_at_5
0,financebench_id_00005,Does Corning have positive working capital bas...,incorrect,"The model answer is a refusal to answer, while...",supported,valid,not_applicable,The model answer states the context lacks info...,0.000000,0,0,0
1,financebench_id_00070,Does American Water Works have positive workin...,incorrect,The model answer fails to provide the required...,supported,valid,not_applicable,The model answer states the context lacks info...,0.000000,1,1,1
2,financebench_id_00080,Does Paypal have positive working capital base...,incorrect,"The model answer is a refusal to answer, while...",insufficient_context,not_applicable,not_applicable,The retrieved context does not contain the nec...,0.000000,0,0,0
3,financebench_id_00206,Are JPM's gross margins historically consisten...,incorrect,The model answer fails to address the question...,supported,valid,not_applicable,The model answer states the context lacks suff...,0.000000,0,0,0
4,financebench_id_00215,Is Verizon a capital intensive business based ...,correct,The model answer correctly identifies Verizon ...,supported,valid,valid,The answer's numeric claim and citation are su...,0.714286,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...
95,financebench_id_02024,"As of FY 2021, how much did Verizon expect to ...",incorrect,The model answer incorrectly states insufficie...,insufficient_context,not_applicable,not_applicable,The retrieved context does not contain any inf...,NaN,0,0,0
96,financebench_id_02049,"Looking at VaR, did the risk that JPM faced in...",correct,The model answer is semantically equivalent to...,supported,valid,valid,The retrieved context from JPMORGAN_2023Q2_10Q...,NaN,0,1,1
97,financebench_id_02119,If JPM went bankrupted by the end by 2021 Q1 a...,incorrect,"The model answer is a refusal to answer, while...",insufficient_context,not_applicable,not_applicable,The retrieved context lacks JPM's total shares...,NaN,0,0,0
98,financebench_id_02416,What are three main companies acquired by Pfiz...,incorrect,"The model answer is a refusal, not a factual a...",insufficient_context,missing,not_applicable,The retrieved context contains no substantive ...,NaN,0,0,0


,metric,value,n
0,average_correctness,0.240000,100
1,average_faithfulness_first_20,0.395714,20
2,support_supported_rate,0.420000,100
3,support_valid_citation_rate,0.470000,100
4,page_hit_at_1,0.200000,100
5,page_hit_at_3,0.330000,100
6,page_hit_at_5,0.400000,100


### Task 6 Results

The evaluation file is written to `rag_faiss/outputs/assignment_2_evaluation.xlsx` with one row per question in the filtered FinanceBench dataset. This saved run uses `deepseek-ai/DeepSeek-V3.2` as the judge model, and the raw Task 6 artifacts record that same configured judge model. Faithfulness is computed with `ragas.metrics.collections.Faithfulness.score(...)` on the first 20 sorted examples only, using the same configured DeepSeek judge model through `ragas.llms.llm_factory` and an `AsyncOpenAI` Nebius client.

Retrieval is reported as strict `page_hit_at_1`, `page_hit_at_3`, and `page_hit_at_5`. A hit requires that a retrieved chunk's `doc_name` and 0-indexed `page_number` match one of the dataset evidence pages. The RAG baseline used for this evaluation retrieves with the question text only; gold dataset metadata such as the expected `doc_name`, `company`, or `doc_period` is used only after retrieval for evaluation, not as retriever or prompt input.

Saved aggregate results from `rag_faiss/outputs/assignment_2_evaluation.xlsx`:

| metric | value | rows |
|---|---:|---:|
| average correctness | 0.240 | 100 |
| average faithfulness | 0.396 | 20 |
| support supported rate | 0.420 | 100 |
| support valid citation rate | 0.470 | 100 |
| page-hit@1 | 0.200 | 100 |
| page-hit@3 | 0.330 | 100 |
| page-hit@5 | 0.400 | 100 |

The main bottleneck remains retrieval: only 40 of 100 questions retrieved a chunk from the expected document and evidence page within the top five. The support verifier judged 42 answers as fully supported, while the correctness judge accepted 24 answers, showing that supported-looking answers can still miss the FinanceBench gold answer or required calculation.


## Task 7 - Improvement Cycles

Baseline is the Task 6 query-only pipeline: FAISS retrieval with `BAAI/bge-small-en-v1.5`, top-5 chunks sent to `Llama-3.3-70B-Instruct`, the original concise citation prompt, correctness judged with `DeepSeek-V3.2`, Ragas faithfulness on the first 20 sorted questions, and retrieval `page-hit@k` on the full filtered dataset.

**Experiment 1 - remove few-shot examples.** I expect answer-format consistency and possibly correctness to drop because the model loses examples for numeric answers and insufficient-context refusals. Retrieval hit-rate should not move because the retrieved chunks are unchanged.

**Experiment 2 - stricter generation prompt.** I expect support/citation behavior and correctness to improve if the model follows stricter evidence rules, though it may refuse more often when the retrieved context is incomplete. Retrieval hit-rate should not move because the retrieved chunks are unchanged.

**Experiment 3 - larger generator context (`k=8`).** I expect `page-hit@8` and possibly correctness to improve because more retrieved chunks create more chances to include the evidence page. Faithfulness or support may drop if the extra chunks introduce distracting context.

**Experiment 4 - BGE reranker.** I expect `page-hit@1`/`page-hit@3`/`page-hit@4` and correctness to improve if a cross-encoder reranker orders the initial FAISS top-20 candidates more precisely for the query. The reranker sends the top-4 reranked chunks to the generator.


In [17]:
TASK7_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

required_task7_globals = [
    "task6_df",
    "rag_vectorstore",
    "RAG_SYSTEM_PROMPT",
    "RAG_GENERATION_MODEL",
    "NEBIUS_BASE_URL",
    "TASK6_JUDGE_MODEL",
    "TASK6_JUDGE_RUN_VERSION",
    "TASK6_SUPPORT_RUN_VERSION",
    "RAG_FEW_SHOT_MESSAGES",
    "get_nebius_client",
    "build_rag_user_prompt",
    "build_rag_messages",
    "judge_correctness",
    "has_valid_correctness_record",
    "judge_answer_support",
    "has_valid_support_record",
    "has_valid_faithfulness_record",
    "score_faithfulness_with_sync_api",
    "build_contexts_from_chunks",
    "get_expected_pages",
    "page_hit_from_chunks",
]
missing_task7_globals = [name for name in required_task7_globals if name not in globals()]
if missing_task7_globals:
    raise RuntimeError(f"Run the Task 4 and Task 6 cells before Task 7. Missing: {missing_task7_globals}")


def task7_save_json(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(rows, indent=2, ensure_ascii=False))


def task7_load_json(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = json.loads(path.read_text())
    if not isinstance(rows, list):
        raise ValueError(f"Expected a JSON list in {path}, found {type(rows).__name__}")
    return rows


def task7_artifact_path(experiment: str, kind: str) -> Path:
    return TASK7_ARTIFACT_DIR / f"{experiment}_{kind}.json"


def task7_serialize_doc(doc, rank: int) -> dict:
    metadata = doc.metadata
    return {
        "rank": rank,
        "doc_name": metadata.get("doc_name"),
        "company": metadata.get("company"),
        "doc_period": metadata.get("doc_period"),
        "page_number": metadata.get("page_number"),
        "content": doc.page_content,
    }


def task7_format_context(chunks: list[dict]) -> str:
    if not chunks:
        return "NO_RELEVANT_CONTEXT_RETRIEVED"
    formatted = []
    for rank, chunk in enumerate(chunks, start=1):
        formatted.append(
            f"--- Chunk {rank} ---\n"
            f"doc_name: {chunk.get('doc_name', 'unknown_doc')}\n"
            f"company: {chunk.get('company', 'unknown_company')}\n"
            f"doc_period: {chunk.get('doc_period', 'unknown_period')}\n"
            f"page_number: {chunk.get('page_number', 'unknown_page')}\n"
            f"content:\n{' '.join(str(chunk.get('content', '')).split())}"
        )
    return "\n\n".join(formatted)


def task7_generate_from_chunks(
    client,
    query: str,
    chunks: list[dict],
    system_prompt: str,
    model: str,
    few_shot_messages: "list[dict] | None" = None,
) -> str:
    context_block = task7_format_context(chunks)
    user_prompt = build_rag_user_prompt(query, context_block)

    def _request():
        response = client.chat.completions.create(
            model=model,
            messages=build_rag_messages(system_prompt, user_prompt, few_shot_messages=few_shot_messages),
            temperature=0,
            max_tokens=700,
        )
        return response.choices[0].message.content.strip()

    return call_with_retries(_request, "Task 7 generation")


baseline_rag_by_id = {row["financebench_id"]: row for row in rag_rows}
if len(baseline_rag_by_id) != len(task6_df):
    raise RuntimeError("Task 6 RAG rows are incomplete. Re-run Task 6 before Task 7.")

_task7_reranker_lock = Lock()


def task7_retrieve_for_experiment(row: pd.Series, experiment_config: dict, reranker=None) -> list[dict]:
    mode = experiment_config["retrieval_mode"]
    if mode == "baseline_chunks":
        chunks = baseline_rag_by_id[row["financebench_id"]]["retrieved_chunks"]
        return [dict(chunk, rank=rank) for rank, chunk in enumerate(chunks[: experiment_config["final_k"]], start=1)]

    if mode == "faiss":
        docs = rag_vectorstore.similarity_search(row["question"], k=experiment_config["retrieval_k"])
        return [task7_serialize_doc(doc, rank) for rank, doc in enumerate(docs, start=1)]

    if mode == "reranker":
        candidate_docs = rag_vectorstore.similarity_search(row["question"], k=experiment_config["candidate_k"])
        if reranker is None:
            reranker = CrossEncoder(experiment_config["reranker_model"], max_length=512, device=LOCAL_TORCH_DEVICE)
        pairs = [[row["question"], doc.page_content] for doc in candidate_docs]
        with _task7_reranker_lock:
            scores = reranker.predict(pairs, show_progress_bar=False)
        ranked = sorted(zip(candidate_docs, scores), key=lambda item: float(item[1]), reverse=True)
        top_docs = [doc for doc, _ in ranked[: experiment_config["final_k"]]]
        return [task7_serialize_doc(doc, rank) for rank, doc in enumerate(top_docs, start=1)]

    raise ValueError(f"Unknown retrieval mode: {mode}")


def task7_run_rag_experiment(experiment_config: dict) -> list[dict]:
    path = task7_artifact_path(experiment_config["experiment"], "rag_answers_raw")
    client = get_nebius_client()
    reranker = None
    if experiment_config["retrieval_mode"] == "reranker":
        reranker = CrossEncoder(experiment_config["reranker_model"], max_length=512, device=LOCAL_TORCH_DEVICE)

    def _row(item: tuple[int, pd.Series]) -> dict:
        _, row = item
        financebench_id = row["financebench_id"]
        retrieved_chunks = task7_retrieve_for_experiment(row, experiment_config, reranker=reranker)
        rag_answer = task7_generate_from_chunks(
            client=client,
            query=row["question"],
            chunks=retrieved_chunks,
            system_prompt=experiment_config["system_prompt"],
            model=experiment_config["generation_model"],
            few_shot_messages=experiment_config.get("few_shot_messages"),
        )
        return {
            "financebench_id": financebench_id,
            "question": row["question"],
            "answer": row["answer"],
            "rag_answer": rag_answer,
            "retrieved_chunks": retrieved_chunks,
            "retrieved_doc_names": [chunk.get("doc_name") for chunk in retrieved_chunks],
            "retrieved_page_numbers": [chunk.get("page_number") for chunk in retrieved_chunks],
        }

    rows = run_parallel_jobs(
        task6_df.iterrows(),
        _row,
        desc=f"Task 7 RAG {experiment_config['experiment']}",
        max_workers=API_GENERATION_WORKERS,
        checkpoint_path=path,
    )
    task7_save_json(path, rows)
    return rows


def task7_run_correctness(experiment: str, rag_rows: list[dict]) -> list[dict]:
    path = task7_artifact_path(experiment, "correctness_raw")
    rag_by_id = {row["financebench_id"]: row for row in rag_rows}
    ensure_api_ca_bundle()
    judge_client = OpenAI(base_url=NEBIUS_BASE_URL, api_key=os.environ["NEBIUS_API_KEY"], timeout=60.0, max_retries=API_MAX_RETRIES)

    def _row(item: tuple[int, pd.Series]) -> dict:
        _, row = item
        financebench_id = row["financebench_id"]
        verdict = judge_correctness(
            judge_client,
            row["question"],
            row["answer"],
            rag_by_id[financebench_id]["rag_answer"],
        )
        return {
            "financebench_id": financebench_id,
            "judge_model": TASK6_JUDGE_MODEL,
            "judge_run_version": TASK6_JUDGE_RUN_VERSION,
            "correctness": verdict["verdict"],
            "correctness_justification": verdict["justification"],
        }

    rows = run_parallel_jobs(
        task6_df.iterrows(),
        _row,
        desc=f"Task 7 judge {experiment}",
        max_workers=API_JUDGE_WORKERS,
        checkpoint_path=path,
    )
    by_id = {row["financebench_id"]: row for row in rows}

    missing = [
        row["financebench_id"]
        for _, row in task6_df.iterrows()
        if not has_valid_correctness_record(by_id.get(row["financebench_id"]))
    ]
    if missing:
        raise RuntimeError(f"Task 7 correctness records for {experiment} are incomplete or invalid for {len(missing)} rows: {missing[:5]}")

    ordered = [by_id[row["financebench_id"]] for _, row in task6_df.iterrows()]
    task7_save_json(path, ordered)
    return ordered


def task7_run_support(experiment: str, rag_rows: list[dict]) -> list[dict]:
    path = task7_artifact_path(experiment, "support_raw")
    rag_by_id = {row["financebench_id"]: row for row in rag_rows}
    ensure_api_ca_bundle()
    judge_client = OpenAI(base_url=NEBIUS_BASE_URL, api_key=os.environ["NEBIUS_API_KEY"], timeout=60.0, max_retries=API_MAX_RETRIES)

    def _row(item: tuple[int, pd.Series]) -> dict:
        _, row = item
        financebench_id = row["financebench_id"]
        rag_record = rag_by_id[financebench_id]
        support = judge_answer_support(
            judge_client,
            row["question"],
            rag_record["rag_answer"],
            rag_record["retrieved_chunks"],
        )
        return {
            "financebench_id": financebench_id,
            "judge_model": TASK6_JUDGE_MODEL,
            "support_run_version": TASK6_SUPPORT_RUN_VERSION,
            "support_verdict": support["support_verdict"],
            "citation_status": support["citation_status"],
            "numeric_status": support["numeric_status"],
            "support_justification": support["justification"],
        }

    rows = run_parallel_jobs(
        task6_df.iterrows(),
        _row,
        desc=f"Task 7 support {experiment}",
        max_workers=API_JUDGE_WORKERS,
        checkpoint_path=path,
    )
    by_id = {row["financebench_id"]: row for row in rows}

    missing = [
        row["financebench_id"]
        for _, row in task6_df.iterrows()
        if not has_valid_support_record(by_id.get(row["financebench_id"]))
    ]
    if missing:
        raise RuntimeError(f"Task 7 support records for {experiment} are incomplete or invalid for {len(missing)} rows: {missing[:5]}")

    ordered = [by_id[row["financebench_id"]] for _, row in task6_df.iterrows()]
    task7_save_json(path, ordered)
    return ordered


def task7_run_faithfulness(experiment: str, rag_rows: list[dict]) -> list[dict]:
    path = task7_artifact_path(experiment, "faithfulness_raw")
    rag_by_id = {row["financebench_id"]: row for row in rag_rows}
    faithfulness_eval_df = task6_df.head(TASK7_FAITHFULNESS_LIMIT)

    def _row(item: tuple[int, pd.Series]) -> dict:
        _, row = item
        financebench_id = row["financebench_id"]
        rag_record = rag_by_id[financebench_id]
        retrieved_contexts = build_contexts_from_chunks(rag_record["retrieved_chunks"], query=row["question"])
        try:
            score = score_faithfulness_with_sync_api(
                user_input=row["question"],
                response=rag_record["rag_answer"],
                retrieved_contexts=retrieved_contexts,
            )
            error = None
        except Exception as exc:
            score = float("nan")
            error = repr(exc)
        return {
            "financebench_id": financebench_id,
            "judge_model": TASK6_JUDGE_MODEL,
            "judge_run_version": TASK6_JUDGE_RUN_VERSION,
            "faithfulness": score,
            "faithfulness_error": error,
        }

    rows = run_parallel_jobs(
        faithfulness_eval_df.iterrows(),
        _row,
        desc=f"Task 7 faithfulness {experiment}",
        max_workers=API_FAITHFULNESS_WORKERS,
        checkpoint_path=path,
    )
    by_id = {row["financebench_id"]: row for row in rows}

    missing = [
        row["financebench_id"]
        for _, row in faithfulness_eval_df.iterrows()
        if not has_valid_faithfulness_record(by_id.get(row["financebench_id"]))
    ]
    if missing:
        raise RuntimeError(f"Task 7 faithfulness records for {experiment} are incomplete or invalid for {len(missing)} rows: {missing[:5]}")

    ordered = [by_id[row["financebench_id"]] for _, row in faithfulness_eval_df.iterrows()]
    task7_save_json(path, ordered)
    return ordered


def task7_page_hit_means(rag_rows: list[dict], k_values: list[int]) -> dict:
    rag_by_id = {row["financebench_id"]: row for row in rag_rows}
    values = {f"page_hit_at_{k}": [] for k in k_values}
    for _, row in task6_df.iterrows():
        retrieved_chunks = rag_by_id[row["financebench_id"]]["retrieved_chunks"]
        expected_pages = get_expected_pages(row)
        for k in k_values:
            values[f"page_hit_at_{k}"].append(
                page_hit_from_chunks(retrieved_chunks, row["doc_name"], expected_pages, k)
            )
    return {metric: sum(metric_values) / len(metric_values) for metric, metric_values in values.items()}


def task7_summarize_experiment(
    experiment_config: dict,
    rag_rows: list[dict],
    correctness_rows: list[dict],
    support_rows: list[dict],
    faithfulness_rows: list[dict],
) -> dict:
    correctness = sum(row["correctness"] == "correct" for row in correctness_rows) / len(correctness_rows)
    support_supported_rate = sum(row["support_verdict"] == "supported" for row in support_rows) / len(support_rows)
    support_valid_citation_rate = sum(row["citation_status"] == "valid" for row in support_rows) / len(support_rows)
    faithfulness_values = []
    for row in faithfulness_rows:
        value = row.get("faithfulness")
        try:
            value = float(value)
        except Exception:
            continue
        if not math.isnan(value):
            faithfulness_values.append(value)
    faithfulness = sum(faithfulness_values) / len(faithfulness_values) if faithfulness_values else float("nan")

    summary = {
        "experiment": experiment_config["experiment"],
        "change": experiment_config["change"],
        "correctness": correctness,
        "support_supported_rate": support_supported_rate,
        "support_valid_citation_rate": support_valid_citation_rate,
        "faithfulness": faithfulness,
    }
    summary.update(task7_page_hit_means(rag_rows, experiment_config["page_hit_k_values"]))
    return summary


def task7_build_baseline_row() -> dict:
    task6_eval = pd.read_excel(TASK6_XLSX_PATH)
    row = {
        "experiment": TASK7_BASELINE_NAME,
        "change": "Task 6 baseline: query-only FAISS top-5, original prompt, Llama-3.3-70B-Instruct.",
        "correctness": (task6_eval["correctness"] == "correct").mean(),
        "faithfulness": task6_eval.loc[task6_eval["faithfulness"].notna(), "faithfulness"].mean(),
    }
    if "support_verdict" in task6_eval.columns:
        row["support_supported_rate"] = (task6_eval["support_verdict"] == "supported").mean()
    else:
        row["support_supported_rate"] = pd.NA
    if "support_citation_status" in task6_eval.columns:
        row["support_valid_citation_rate"] = (task6_eval["support_citation_status"] == "valid").mean()
    else:
        row["support_valid_citation_rate"] = pd.NA
    for k in TASK7_BASELINE_PAGE_HIT_K_VALUES:
        row[f"page_hit_at_{k}"] = task6_eval[f"page_hit_at_{k}"].mean()
    row["page_hit_at_4"] = task7_page_hit_means(rag_rows, [4])["page_hit_at_4"]
    return row


def task7_has_complete_rag_rows(rows: list[dict], expected_ids: list[str]) -> bool:
    if len(rows) != len(expected_ids):
        return False
    by_id = {row.get("financebench_id"): row for row in rows if isinstance(row, dict)}
    if set(by_id) != set(expected_ids):
        return False
    return all(
        isinstance(by_id[financebench_id].get("rag_answer"), str)
        and isinstance(by_id[financebench_id].get("retrieved_chunks"), list)
        for financebench_id in expected_ids
    )


def task7_records_match_expected_ids(rows: list[dict], expected_ids: list[str], validator) -> bool:
    if len(rows) != len(expected_ids):
        return False
    by_id = {row.get("financebench_id"): row for row in rows if isinstance(row, dict)}
    if set(by_id) != set(expected_ids):
        return False
    return all(validator(by_id[financebench_id]) for financebench_id in expected_ids)


def task7_order_rows(rows: list[dict], expected_ids: list[str]) -> list[dict]:
    by_id = {row["financebench_id"]: row for row in rows}
    return [by_id[financebench_id] for financebench_id in expected_ids]


def task7_load_cached_experiment_outputs(experiment_config: dict):
    if FORCE_TASK7_REGEN:
        return None

    expected_ids = task6_df["financebench_id"].tolist()
    faithfulness_expected_ids = task6_df.head(TASK7_FAITHFULNESS_LIMIT)["financebench_id"].tolist()

    rag_rows = task7_load_json(task7_artifact_path(experiment_config["experiment"], "rag_answers_raw"))
    correctness_rows = task7_load_json(task7_artifact_path(experiment_config["experiment"], "correctness_raw"))
    support_rows = task7_load_json(task7_artifact_path(experiment_config["experiment"], "support_raw"))
    faithfulness_rows = task7_load_json(task7_artifact_path(experiment_config["experiment"], "faithfulness_raw"))

    if not task7_has_complete_rag_rows(rag_rows, expected_ids):
        return None
    if not task7_records_match_expected_ids(correctness_rows, expected_ids, has_valid_correctness_record):
        return None
    if not task7_records_match_expected_ids(support_rows, expected_ids, has_valid_support_record):
        return None
    if not task7_records_match_expected_ids(faithfulness_rows, faithfulness_expected_ids, has_valid_faithfulness_record):
        return None

    return {
        "rag_rows": task7_order_rows(rag_rows, expected_ids),
        "correctness_rows": task7_order_rows(correctness_rows, expected_ids),
        "support_rows": task7_order_rows(support_rows, expected_ids),
        "faithfulness_rows": task7_order_rows(faithfulness_rows, faithfulness_expected_ids),
    }


summary_rows = [task7_build_baseline_row()]
experiment_outputs = {}

for experiment_config in TASK7_EXPERIMENTS:
    experiment_name = experiment_config["experiment"]
    cached_outputs = task7_load_cached_experiment_outputs(experiment_config)
    if cached_outputs is not None:
        print(f"Loaded cached Task 7 outputs for {experiment_name}")
        rag_rows = cached_outputs["rag_rows"]
        correctness_rows = cached_outputs["correctness_rows"]
        support_rows = cached_outputs["support_rows"]
        faithfulness_rows = cached_outputs["faithfulness_rows"]
    else:
        load_dotenv(REPO_ROOT / ".env")
        if not os.getenv("NEBIUS_API_KEY"):
            raise ValueError("NEBIUS_API_KEY is not set. Add it to .env before regenerating Task 7 artifacts.")
        rag_rows = task7_run_rag_experiment(experiment_config)
        correctness_rows = task7_run_correctness(experiment_name, rag_rows)
        support_rows = task7_run_support(experiment_name, rag_rows)
        faithfulness_rows = task7_run_faithfulness(experiment_name, rag_rows)
    summary_rows.append(task7_summarize_experiment(experiment_config, rag_rows, correctness_rows, support_rows, faithfulness_rows))
    experiment_outputs[experiment_name] = {
        "rag_rows": rag_rows,
        "correctness_rows": correctness_rows,
        "support_rows": support_rows,
        "faithfulness_rows": faithfulness_rows,
    }

assignment_2_improvement_cycles = pd.DataFrame(summary_rows)
task7_output_columns = [
    "experiment",
    "change",
    "correctness",
    "support_supported_rate",
    "support_valid_citation_rate",
    "faithfulness",
    "page_hit_at_1",
    "page_hit_at_3",
    "page_hit_at_4",
    "page_hit_at_5",
    "page_hit_at_8",
]
assignment_2_improvement_cycles = assignment_2_improvement_cycles.reindex(columns=task7_output_columns)
assert int(assignment_2_improvement_cycles["correctness"].notna().sum()) == len(assignment_2_improvement_cycles)
assert int(assignment_2_improvement_cycles["faithfulness"].notna().sum()) == len(assignment_2_improvement_cycles)
assignment_2_improvement_cycles.to_excel(TASK7_OUTPUT_XLSX_PATH, index=False)
print(f"Saved Task 7 deliverable to {TASK7_OUTPUT_XLSX_PATH}")
display(assignment_2_improvement_cycles)


Loaded cached Task 7 outputs for prompt_no_few_shot
Loaded cached Task 7 outputs for strict_prompt
Loaded cached Task 7 outputs for top_k_8
Loaded cached Task 7 outputs for bge_reranker_top4
Saved Task 7 deliverable to rag_faiss/outputs/assignment_2_improvement_cycles.xlsx


,experiment,change,correctness,support_supported_rate,support_valid_citation_rate,faithfulness,page_hit_at_1,page_hit_at_3,page_hit_at_4,page_hit_at_5,page_hit_at_8
0,task6_baseline,"Task 6 baseline: query-only FAISS top-5, origi...",0.24,0.42,0.47,0.395714,0.20,0.33,0.36,0.4,NaN
1,prompt_no_few_shot,A/B prompt ablation: reused Task 6 top-5 chunk...,0.28,0.51,0.58,0.455330,0.20,0.33,0.36,0.4,NaN
2,strict_prompt,Changed only the generation system prompt; reu...,0.24,0.42,0.48,0.361190,0.20,0.33,0.36,0.4,NaN
3,top_k_8,Changed only the number of FAISS chunks sent t...,0.30,0.38,0.48,0.469306,0.20,0.33,0.36,0.4,0.45
4,bge_reranker_top4,Added BAAI/bge-reranker-base: retrieve top-20 ...,0.23,0.34,0.43,0.332917,0.13,0.22,0.27,NaN,NaN


### Task 7 Results and Interpretation

This saved run loaded the cached experiment artifacts and wrote `rag_faiss/outputs/assignment_2_improvement_cycles.xlsx` with the Task 6 baseline followed by the prompt ablation, strict prompt, top-k increase, and BGE reranker experiments. All experiments used the same 100-question filtered dataset for correctness/support/page-hit metrics and the first 20 sorted questions for faithfulness.

| experiment | correctness | support | valid citation | faithfulness | retrieval result |
|---|---:|---:|---:|---:|---|
| Task 6 baseline | 0.24 | 0.42 | 0.47 | 0.396 | page-hit@5 = 0.40 |
| no few-shot examples | 0.28 | 0.51 | 0.58 | 0.455 | page-hit@5 = 0.40 |
| strict prompt | 0.24 | 0.42 | 0.48 | 0.361 | page-hit@5 = 0.40 |
| top-k 8 | 0.30 | 0.38 | 0.48 | 0.469 | page-hit@5 = 0.40, page-hit@8 = 0.45 |
| BGE reranker top-4 | 0.23 | 0.34 | 0.43 | 0.333 | page-hit@4 = 0.27 |

**Experiment 1 - no few-shot examples.** This hypothesis was wrong: removing the few-shot examples improved every generation-facing metric while retrieval stayed unchanged. Correctness rose from 0.24 to 0.28, support from 0.42 to 0.51, valid citation rate from 0.47 to 0.58, and faithfulness from 0.396 to 0.455. In this setup the few-shot examples seem to have over-constrained the answer style more than they helped reasoning.

**Experiment 2 - strict prompt.** This hypothesis was not supported. The stricter instructions did not improve correctness or support versus baseline, valid citation rate only moved slightly from 0.47 to 0.48, and faithfulness fell from 0.396 to 0.361. The prompt seems to have changed answer style more than answer quality.

**Experiment 3 - top-k 8.** This mostly matched the hypothesis on recall and answer quality. `page-hit@8` improved to 0.45, correctness rose to the best score at 0.30, and faithfulness also reached the best score at 0.469. But support fell from 0.42 to 0.38, so the extra chunks helped recall and final answers while also introducing distractors that weakened grounding.

**Experiment 4 - BGE reranker top-4.** This hypothesis was not supported. The reranker hurt every tracked metric in this run: correctness fell to 0.23, support to 0.34, valid citation rate to 0.43, faithfulness to 0.333, and retrieval dropped to `page-hit@4 = 0.27` with weaker `page-hit@1` and `page-hit@3` as well. The reranker likely preferred semantically similar chunks that sounded relevant without actually landing on the dataset evidence page.

**Wrap-up.** Retrieval is still the larger bottleneck: baseline `page-hit@5` is only 0.40, so in most questions the generator never sees the exact evidence page. But these saved experiment results also show that generation settings matter materially: removing the few-shot examples improved correctness from 0.24 to 0.28 without changing retrieval, and increasing context to `k=8` pushed correctness to 0.30 at the cost of weaker support. If I had one more week, I would focus first on retrieval with metadata-aware filtering for company and filing period, table-aware chunking for numeric questions, and a stronger retrieval stack such as hybrid search plus a better-tuned reranker; after that I would add lightweight calculation/post-processing for numeric answers.


## Bonus - Multi-Scale Chunking

The AI21 article in `references/article.md` argues that chunk size is query-dependent: smaller chunks can preserve specific details, while larger chunks can preserve broader context, so a single fixed size may underperform for some queries. Here I test the idea on the filtered FinanceBench set using the stricter assignment metric, `page-hit@5`, across three chunk sizes: `500`, `1000`, and `2000`. The `1000` index is the Task 3 baseline; the `500` and `2000` indices are built with the same `PyPDFLoader`, metadata, `RecursiveCharacterTextSplitter`, fixed `chunk_overlap=150`, and `BAAI/bge-small-en-v1.5` embedding model.


In [18]:
BONUS_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

bonus_df = pd.read_json(BONUS_DATASET_PATH, lines=True)
bonus_df = bonus_df.sort_values("financebench_id", kind="stable").reset_index(drop=True)
bonus_relevant_docs = (
    bonus_df[["doc_name", "company", "doc_period"]]
    .drop_duplicates(subset=["doc_name"])
    .sort_values("doc_name", kind="stable")
    .reset_index(drop=True)
)
bonus_relevant_docs["repo_pdf_filename"] = bonus_relevant_docs["doc_name"].map(lambda doc_name: f"{doc_name}.pdf")

bonus_embedding_model = HuggingFaceEmbeddings(
    model_name=BONUS_EMBEDDING_MODEL_NAME,
    model_kwargs={"device": LOCAL_TORCH_DEVICE},
    encode_kwargs={"normalize_embeddings": True},
    show_progress=True,
)

def bonus_as_list(value):
    if isinstance(value, list):
        return value
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except Exception:
        pass
    return [value]

def bonus_expected_pages(row: pd.Series) -> set[int]:
    return {int(page) for page in bonus_as_list(row.get("evidence_page_nums")) if str(page).strip() != ""}

def bonus_index_is_ready(index_dir: Path, chunk_size: int) -> bool:
    manifest_path = index_dir / "manifest.json"
    if not (index_dir / "index.faiss").exists() or not (index_dir / "index.pkl").exists() or not manifest_path.exists():
        return False
    manifest = json.loads(manifest_path.read_text())
    return (
        manifest.get("embedding_model") == BONUS_EMBEDDING_MODEL_NAME
        and int(manifest.get("chunk_size", -1)) == int(chunk_size)
        and int(manifest.get("chunk_overlap", -1)) == int(BONUS_CHUNK_OVERLAP)
    )

def bonus_load_pages() -> list:
    pages = []
    load_errors = []
    for _, row in tqdm(bonus_relevant_docs.iterrows(), total=len(bonus_relevant_docs), desc="Bonus loading PDF pages"):
        pdf_path = BONUS_PDF_DIR / row["repo_pdf_filename"]
        if not pdf_path.exists():
            load_errors.append({"doc_name": row["doc_name"], "pdf_path": str(pdf_path), "error": "missing PDF"})
            continue
        try:
            loaded_pages = PyPDFLoader(str(pdf_path)).load()
        except Exception as exc:
            load_errors.append({"doc_name": row["doc_name"], "pdf_path": str(pdf_path), "error": repr(exc)})
            continue

        for fallback_page_number, page in enumerate(loaded_pages):
            page_number = int(page.metadata.get("page", fallback_page_number))
            page.metadata.update(
                {
                    "doc_name": row["doc_name"],
                    "company": row["company"],
                    "doc_period": int(row["doc_period"]),
                    "page_number": page_number,
                }
            )
            page.metadata.pop("page", None)
            pages.append(page)

    if load_errors:
        display(pd.DataFrame(load_errors))
        raise RuntimeError("At least one PDF failed to load for the bonus indices.")

    print(f"Loaded bonus PDF pages: {len(pages):,}", flush=True)
    return pages

def bonus_build_or_load_index(chunk_size: int, pages: list) -> FAISS:
    index_dir = BONUS_INDEX_DIRS[chunk_size]
    manifest_path = index_dir / "manifest.json"
    if bonus_index_is_ready(index_dir, chunk_size) and not BONUS_REBUILD_INDICES:
        print(f"Loaded existing chunk_size={chunk_size} index from {index_dir}", flush=True)
        return FAISS.load_local(str(index_dir), bonus_embedding_model, allow_dangerous_deserialization=True)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=BONUS_CHUNK_OVERLAP,
    )
    chunks = splitter.split_documents(pages)
    print(f"Building chunk_size={chunk_size} index from {len(pages):,} pages and {len(chunks):,} chunks", flush=True)

    vectorstore = FAISS.from_documents(chunks, bonus_embedding_model)
    index_dir.mkdir(parents=True, exist_ok=True)
    vectorstore.save_local(str(index_dir))

    manifest = {
        "embedding_model": BONUS_EMBEDDING_MODEL_NAME,
        "chunk_size": int(chunk_size),
        "chunk_overlap": int(BONUS_CHUNK_OVERLAP),
        "filtered_dataset_rows": int(len(bonus_df)),
        "pdf_count": int(len(bonus_relevant_docs)),
        "page_count": int(len(pages)),
        "chunk_count": int(len(chunks)),
    }
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print(f"Saved chunk_size={chunk_size} index to {index_dir}", flush=True)
    print(json.dumps(manifest, indent=2), flush=True)
    return vectorstore

def bonus_page_hit_from_docs(docs: list, expected_doc: str, expected_pages: set[int]) -> int:
    if not expected_pages:
        return 0
    for doc in docs[:5]:
        metadata = doc.metadata
        if metadata.get("doc_name") == expected_doc and metadata.get("page_number") in expected_pages:
            return 1
    return 0

def bonus_save_json(path: Path, rows: list[dict]) -> None:
    path.write_text(json.dumps(rows, indent=2, ensure_ascii=False))

bonus_pages = bonus_load_pages()
bonus_vectorstores = {}
for chunk_size in BONUS_CHUNK_SIZES:
    bonus_vectorstores[chunk_size] = bonus_build_or_load_index(chunk_size, bonus_pages)

# Keep index-building progress visible, but avoid a one-batch progress bar for every query.
bonus_embedding_model.show_progress = False

per_size_rows = {}
for chunk_size in BONUS_CHUNK_SIZES:
    artifact_path = BONUS_ARTIFACT_DIR / f"page_hit_at5_chunk{chunk_size}.json"
    rows = []
    vectorstore = bonus_vectorstores[chunk_size]

    for _, row in tqdm(bonus_df.iterrows(), total=len(bonus_df), desc=f"Bonus page-hit@5 chunk_size={chunk_size}"):
        financebench_id = row["financebench_id"]
        docs = vectorstore.similarity_search(row["question"], k=5)
        retrieved = [
            {
                "rank": rank,
                "doc_name": doc.metadata.get("doc_name"),
                "page_number": doc.metadata.get("page_number"),
                "company": doc.metadata.get("company"),
                "doc_period": doc.metadata.get("doc_period"),
            }
            for rank, doc in enumerate(docs, start=1)
        ]
        expected_pages = bonus_expected_pages(row)
        raw_row = {
            "financebench_id": financebench_id,
            "doc_name": row["doc_name"],
            "question": row["question"],
            "evidence_page_nums": sorted(expected_pages),
            "chunk_size": int(chunk_size),
            "page_hit_at_5": bonus_page_hit_from_docs(docs, row["doc_name"], expected_pages),
            "retrieved": retrieved,
        }
        rows.append(raw_row)
        bonus_save_json(artifact_path, rows)

    bonus_save_json(artifact_path, rows)
    per_size_rows[chunk_size] = rows

bonus_comparison_rows = []
for _, row in bonus_df.iterrows():
    financebench_id = row["financebench_id"]
    comparison = {
        "financebench_id": financebench_id,
        "question_type": row.get("question_type"),
        "doc_name": row["doc_name"],
        "question": row["question"],
        "evidence_page_nums": row.get("evidence_page_nums"),
    }
    hits = []
    for chunk_size in BONUS_CHUNK_SIZES:
        hit = next(item for item in per_size_rows[chunk_size] if item["financebench_id"] == financebench_id)["page_hit_at_5"]
        comparison[f"page_hit_at_5_chunk{chunk_size}"] = hit
        hits.append(hit)
    comparison["any_chunk_size_hit"] = int(any(hits))
    comparison["all_chunk_sizes_agree"] = int(len(set(hits)) == 1)
    comparison["chunk_size_disagreement"] = int(len(set(hits)) > 1)
    winning_sizes = [size for size, hit in zip(BONUS_CHUNK_SIZES, hits) if hit == max(hits)]
    comparison["best_chunk_sizes"] = ",".join(map(str, winning_sizes))
    comparison["unique_winning_chunk_size"] = winning_sizes[0] if len(winning_sizes) == 1 else pd.NA
    bonus_comparison_rows.append(comparison)

bonus_comparison_df = pd.DataFrame(bonus_comparison_rows)

summary_rows = []
for chunk_size in BONUS_CHUNK_SIZES:
    col = f"page_hit_at_5_chunk{chunk_size}"
    summary_rows.append(
        {
            "chunk_size": chunk_size,
            "page_hit_at_5": bonus_comparison_df[col].mean(),
            "hit_count": int(bonus_comparison_df[col].sum()),
            "unique_winner_count": int((bonus_comparison_df["unique_winning_chunk_size"] == chunk_size).sum()),
        }
    )

summary_rows.append(
    {
        "chunk_size": "oracle_any_size",
        "page_hit_at_5": bonus_comparison_df["any_chunk_size_hit"].mean(),
        "hit_count": int(bonus_comparison_df["any_chunk_size_hit"].sum()),
        "unique_winner_count": int(bonus_comparison_df["chunk_size_disagreement"].sum()),
    }
)
bonus_summary_df = pd.DataFrame(summary_rows)

bonus_disagreement_count = int(bonus_comparison_df["chunk_size_disagreement"].sum())
bonus_disagreement_rate = bonus_disagreement_count / len(bonus_comparison_df)
bonus_unique_winner_count = int(bonus_comparison_df["unique_winning_chunk_size"].notna().sum())
bonus_unique_winner_rate = bonus_unique_winner_count / len(bonus_comparison_df)

with pd.ExcelWriter(BONUS_XLSX_PATH) as writer:
    bonus_summary_df.to_excel(writer, sheet_name="summary", index=False)
    bonus_comparison_df.to_excel(writer, sheet_name="per_question", index=False)

print(f"Saved bonus comparison to {BONUS_XLSX_PATH}")
print(f"Disagreement count: {bonus_disagreement_count}/{len(bonus_comparison_df)} ({bonus_disagreement_rate:.1%})")
print(f"Unique winner count: {bonus_unique_winner_count}/{len(bonus_comparison_df)} ({bonus_unique_winner_rate:.1%})")
display(bonus_summary_df)
display(bonus_comparison_df)


Bonus loading PDF pages:   0%|          | 0/42 [00:00<?, ?it/s]

Bonus loading PDF pages:   2%|▏         | 1/42 [00:27<18:57, 27.74s/it]

Bonus loading PDF pages:   5%|▍         | 2/42 [00:37<11:26, 17.16s/it]

Bonus loading PDF pages:   7%|▋         | 3/42 [00:42<07:30, 11.54s/it]

Bonus loading PDF pages:  10%|▉         | 4/42 [01:06<10:33, 16.67s/it]

...

Bonus page-hit@5 chunk_size=2000:  93%|█████████▎| 93/100 [00:01<00:00, 72.80it/s]

Bonus page-hit@5 chunk_size=2000: 100%|██████████| 100/100 [00:01<00:00, 75.40it/s]

Saved bonus comparison to rag_faiss/outputs/assignment_2_bonus_multiscale_chunking.xlsx
Disagreement count: 30/100 (30.0%)
Unique winner count: 16/100 (16.0%)


,chunk_size,page_hit_at_5,hit_count,unique_winner_count
0,500,0.39,39,5
1,1000,0.40,40,5
2,2000,0.34,34,6
3,oracle_any_size,0.53,53,30


,financebench_id,question_type,doc_name,question,evidence_page_nums,page_hit_at_5_chunk500,page_hit_at_5_chunk1000,page_hit_at_5_chunk2000,any_chunk_size_hit,all_chunk_sizes_agree,chunk_size_disagreement,best_chunk_sizes,unique_winning_chunk_size
0,financebench_id_00005,domain-relevant,CORNING_2022_10K,Does Corning have positive working capital bas...,[59],0,0,0,0,1,0,"500,1000,2000",<NA>
1,financebench_id_00070,domain-relevant,AMERICANWATERWORKS_2022_10K,Does American Water Works have positive workin...,"[80, 81]",1,1,1,1,1,0,"500,1000,2000",<NA>
2,financebench_id_00080,domain-relevant,PAYPAL_2022_10K,Does Paypal have positive working capital base...,[60],0,0,0,0,1,0,"500,1000,2000",<NA>
3,financebench_id_00206,domain-relevant,JPMORGAN_2022_10K,Are JPM's gross margins historically consisten...,[2],0,0,0,0,1,0,"500,1000,2000",<NA>
4,financebench_id_00215,domain-relevant,VERIZON_2022_10K,Is Verizon a capital intensive business based ...,"[22, 55]",0,1,0,1,0,1,1000,1000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,financebench_id_02024,novel-generated,VERIZON_2021_10K,"As of FY 2021, how much did Verizon expect to ...","[62, 93]",1,0,0,1,0,1,500,500
96,financebench_id_02049,novel-generated,JPMORGAN_2023Q2_10Q,"Looking at VaR, did the risk that JPM faced in...",[84],1,1,1,1,1,0,"500,1000,2000",<NA>
97,financebench_id_02119,novel-generated,JPMORGAN_2021Q1_10Q,If JPM went bankrupted by the end by 2021 Q1 a...,[5],0,0,0,0,1,0,"500,1000,2000",<NA>
98,financebench_id_02416,novel-generated,PFIZER_2021_10K,What are three main companies acquired by Pfiz...,"[69, 70]",0,0,0,0,1,0,"500,1000,2000",<NA>


### Bonus Results and Discussion

The bonus comparison was written to `rag_faiss/outputs/assignment_2_bonus_multiscale_chunking.xlsx`. I used three chunk sizes: `500`, `1000`, and `2000`, with the same `RecursiveCharacterTextSplitter`, fixed `chunk_overlap=150`, page metadata, FAISS retriever, and `BAAI/bge-small-en-v1.5` embeddings. In this saved run, the existing indices for all three chunk sizes were loaded from disk: `data/vectorstore/financebench_bge_small_v1_5_chunk500`, `data/vectorstore/financebench_bge_small_v1_5`, and `data/vectorstore/financebench_bge_small_v1_5_chunk2000`.

| chunk size | page-hit@5 | hit count | unique winner count |
|---:|---:|---:|---:|
| 500 | 0.39 | 39 / 100 | 5 |
| 1000 | 0.40 | 40 / 100 | 5 |
| 2000 | 0.34 | 34 / 100 | 6 |
| oracle: any size hits | 0.53 | 53 / 100 | 30 disagreement cases |

The chunk-size results disagreed on `30` out of `100` questions, so the disagreement rate was `30.0%`. There were `16` questions with a single unique winning chunk size: `500` alone won 5 questions, `1000` alone won 5, and `2000` alone won 6. The remaining disagreement cases were partial ties, where more than one size hit but at least one size missed.

There is a weak aggregate winner: `chunk_size=1000` had the best fixed `page-hit@5`, but only by one question over `500` and six questions over `2000`. The oracle score, which counts a hit if any chunk size retrieved the evidence page, reached `0.53`, substantially above the best fixed score of `0.40`. That gap supports the AI21 article's claim in this FinanceBench setting: no single chunk size is optimal for every query, even though `1000` remains the best single default on average.

One caveat is that this test uses evidence-page retrieval, not the article's document-level recall. Page-level retrieval is stricter, especially for long 10-K filings where many pages from the correct document are semantically similar but only a few match the dataset evidence page.
